In [ ]:
# import os
# import pandas as pd
# import pyarrow.parquet as pq

# def carica_universo_estremo_fallback(cartella_sorgente="progetto_trading_data"):
#     if not os.path.exists(cartella_sorgente):
#         print(f"❌ Errore: La cartella '{cartella_sorgente}' non esiste.")
#         return None
        
#     dati_ricaricati = {}
#     print("🔄 Caricamento via PyArrow Table -> Pure Python Dict -> Pandas Constructor...")
    
#     for file_name in os.listdir(cartella_sorgente):
#         # Escludiamo eventuali file di configurazione come il JSON della mappa
#         if file_name.endswith(".parquet"):
#             cluster = file_name.replace(".parquet", "")
#             dati_ricaricati[cluster] = {}
#             file_path = os.path.join(cartella_sorgente, file_name)
            
#             # 1. Leggiamo il file come tabella nativa PyArrow
#             tabella_pyarrow = pq.read_table(file_path)
            
#             # 2. TRICK: Convertiamo in un dizionario standard di Python (liste pure)
#             # Questo evita l'uso di .to_pandas() e distrugge ogni conflitto di librerie
#             dati_dizionario = tabella_pyarrow.to_pydict()
            
#             # 3. Alleviamo il DataFrame passando dal costruttore standard di Pandas
#             df_cluster = pd.DataFrame(dati_dizionario)
            
#             # 4. Smistiamo i ticker nel dizionario per il backtest
#             for ticker, group in df_cluster.groupby('Ticker'):
#                 df_ticker = group.drop(columns=['Ticker']).set_index('time')
#                 df_ticker.index = pd.to_datetime(df_ticker.index)
#                 dati_ricaricati[cluster][ticker] = df_ticker
                
#     tot_ticker = sum(len(t) for t in dati_ricaricati.values())
#     print(f"\n✅ Vittoria! {tot_ticker} ticker pronti in 'dati_puliti' senza alcun conflitto di versione.")
#     return dati_ricaricati

# # Inizializza finalmente la variabile RAM
# dati_puliti = carica_universo_estremo_fallback()

🔄 Caricamento via PyArrow Table -> Pure Python Dict -> Pandas Constructor...

✅ Vittoria! 124 ticker pronti in 'dati_puliti' senza alcun conflitto di versione.


In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore Split 60/40 MACD (Neighborhood Sensitivity Analysis) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE (MACD)
# # ==============================================================================
# fast_range = np.arange(6, 68, 2)     # Fast EMA (Default standard: 12)
# slow_range  = np.arange(20, 100, 4)  # Slow EMA (Default standard: 26)
# sig_range  = np.arange(5, 100, 3)     # Signal Line EMA (Default standard: 9)

# griglia_parametri = [
#     (f, s, sig) 
#     for f in fast_range for s in slow_range for sig in sig_range 
#     if f < s  # La lenta deve essere maggiore della veloce
# ]
# print(f"📦 Griglia tridimensionale generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: COSTRUTTORE SPLIT 60-40 MACD CON SENSITIVITY ANALYSIS
# # ==============================================================================

# # Parametri per l'analisi di sensibilità
# TOLLERANZA_MAX_DEGRADO = 0.25 # Accettiamo massimo un calo del 25% nel vicinato
# MIN_VICINI_VALIDI = 6         # Richiediamo almeno N vicini validi per considerare la zona testabile

# def ottimizza_asset_split_60_40_macd(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_arr = [t[0] for t in griglia]
#     s_arr = [t[1] for t in griglia]
#     sig_arr = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Slow', 'Signal'])
    
#     # Pre-calcolo globale del MACD per eliminare il problema del warmup nell'OOS
#     macd_global = vbt.MACD.run(
#         close, 
#         fast_window=f_arr, 
#         slow_window=s_arr, 
#         signal_window=sig_arr
#     )
    
#     macd_line = macd_global.macd
#     signal_line = macd_global.signal
    
#     macd_line.columns = multi_idx
#     signal_line.columns = multi_idx
    
#     # --- FASE 1: IN-SAMPLE (60% TRAIN) ---
#     close_is = close.iloc[:split_idx].reset_index(drop=True)
    
#     macd_is = macd_line.iloc[:split_idx].reset_index(drop=True)
#     signal_is = signal_line.iloc[:split_idx].reset_index(drop=True)
    
#     entries_raw = macd_is.vbt.crossed_above(signal_is)
#     exits_raw = macd_is.vbt.crossed_below(signal_is)
    
#     entries_is = entries_raw.vbt.signals.fshift(1)
#     exits_is = exits_raw.vbt.signals.fshift(1)
    
#     pf_is = vbt.Portfolio.from_signals(
#         close_is, entries_is, exits_is, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     rets_is = pf_is.returns()
#     std_is = rets_is.std()
    
#     anni_totali_is = len(close_is) / giorni_anno
#     trades_is_array = pf_is.trades.count().reindex(multi_idx, fill_value=0).values
#     tpy_is_array = trades_is_array / anni_totali_is if anni_totali_is > 0 else 0
    
#     # Calcolo Sharpe IS per tutte le combinazioni valide
#     sharpe_array_is = np.where(
#         (std_is > 0) & (tpy_is_array >= 2), 
#         (rets_is.mean() / std_is) * np.sqrt(giorni_anno), 
#         np.nan
#     )

#     # Calcolo Buy & Hold In-Sample
#     rets_bh_is = close_is.pct_change().dropna()
#     vol_bh_is = rets_bh_is.std() * np.sqrt(giorni_anno)
#     sharpe_bh_is = (rets_bh_is.mean() / rets_bh_is.std() * np.sqrt(giorni_anno)) if vol_bh_is > 0 else 0
    
#     # Se non ci sono strategie valide In-Sample, usciamo
#     if np.isnan(sharpe_array_is).all():
#         return "LOW_SHARPE"

#     # --- FASE 2: SENSITIVITY ANALYSIS (Ricerca del primo picco stabile) ---
    
#     # Creiamo una mappa tridimensionale per facilitare la ricerca dei vicini
#     shape_3d = (len(fast_range), len(slow_range), len(sig_range))
#     cubo_sharpe = np.full(shape_3d, np.nan)
    
#     for idx, (f, s, sig) in enumerate(griglia):
#         fi = np.where(fast_range == f)[0][0]
#         si = np.where(slow_range == s)[0][0]
#         sigi = np.where(sig_range == sig)[0][0]
#         cubo_sharpe[fi, si, sigi] = sharpe_array_is[idx]

#     # Ordiniamo gli indici dal miglior Sharpe al peggiore
#     valid_indices = np.where(~np.isnan(sharpe_array_is))[0]
#     sorted_valid_indices = valid_indices[np.argsort(sharpe_array_is[valid_indices])[::-1]]
    
#     best_idx = None
    
#     # Iteriamo sui migliori candidati
#     for candidate_idx in sorted_valid_indices:
#         f, s, sig = griglia[candidate_idx]
#         candidate_sharpe = sharpe_array_is[candidate_idx]
        
#         # Troviamo le coordinate nel cubo
#         fi = np.where(fast_range == f)[0][0]
#         si = np.where(slow_range == s)[0][0]
#         sigi = np.where(sig_range == sig)[0][0]
        
#         # Estraiamo il vicinato (finestra 3x3x3)
#         fi_min, fi_max = max(0, fi-1), min(shape_3d[0], fi+2)
#         si_min, si_max = max(0, si-1), min(shape_3d[1], si+2)
#         sigi_min, sigi_max = max(0, sigi-1), min(shape_3d[2], sigi+2)
        
#         neighborhood = cubo_sharpe[fi_min:fi_max, si_min:si_max, sigi_min:sigi_max]
        
#         # Rimuoviamo la cella centrale e i NaN per valutare solo i vicini validi
#         neighbors_only = neighborhood.flatten()
#         neighbors_only = neighbors_only[~np.isnan(neighbors_only)]
        
#         # Togliamo il candidato stesso per fare la media dei SOLI vicini
#         neighbors_only = neighbors_only[neighbors_only != candidate_sharpe]
        
#         # Controlliamo se ci sono abbastanza vicini per fare una valutazione
#         if len(neighbors_only) < MIN_VICINI_VALIDI:
#             continue # Scartato: è su un bordo isolato o circondato da NaN
            
#         # Calcoliamo la media dello Sharpe del vicinato
#         avg_neighbor_sharpe = np.mean(neighbors_only)
        
#         # Verifica della Tolleranza
#         soglia_minima_accettabile = candidate_sharpe * (1 - TOLLERANZA_MAX_DEGRADO)
        
#         if avg_neighbor_sharpe >= soglia_minima_accettabile:
#             # TROVATO! È il picco più alto che risiede su un altopiano stabile.
#             best_idx = candidate_idx
#             break
            
#     if best_idx is None:
#         return "NO_STABLE_PLATEAU"
        
#     best_f, best_s, best_sig = griglia[best_idx]
#     sharpe_is_scelto = sharpe_array_is[best_idx]
#     data_inizio_oos = close.index[split_idx].strftime('%Y-%m-%d')
    
#     # --- FASE 3: OUT-OF-SAMPLE (40% TEST) ---
#     date_oos = close.iloc[split_idx:].index
#     close_oos = close.iloc[split_idx:].reset_index(drop=True)
    
#     # Estraiamo le linee MACD vincenti già calcolate globalmente
#     best_macd_oos = macd_line[(best_f, best_s, best_sig)].iloc[split_idx:].reset_index(drop=True)
#     best_signal_oos = signal_line[(best_f, best_s, best_sig)].iloc[split_idx:].reset_index(drop=True)
    
#     entries_oos_raw = best_macd_oos.vbt.crossed_above(best_signal_oos)
#     exits_oos_raw = best_macd_oos.vbt.crossed_below(best_signal_oos)
    
#     entries_oos = entries_oos_raw.vbt.signals.fshift(1)
#     exits_oos = exits_oos_raw.vbt.signals.fshift(1)
    
#     pf_oos = vbt.Portfolio.from_signals(
#         close_oos, entries_oos, exits_oos, 
#         freq=None, init_cash=100_000, fees=0.0005, slippage=0.0005
#     )
    
#     # --- FASE 4: METRICHE STRATEGIA VS BUY & HOLD ---
#     equity_returns_oos = pf_oos.returns()
#     equity_returns_oos.index = date_oos
#     equity_curve_oos = (1 + equity_returns_oos).cumprod()
    
#     giorni_totali_oos = len(equity_curve_oos)
#     anni_totali_oos = giorni_totali_oos / giorni_anno
    
#     cagr_strat = (equity_curve_oos.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if anni_totali_oos > 0 else 0
#     vol_strat = equity_returns_oos.std() * np.sqrt(giorni_anno)
#     sharpe_strat = (equity_returns_oos.mean() / equity_returns_oos.std() * np.sqrt(giorni_anno)) if vol_strat > 0 else 0
    
#     peak_strat = equity_curve_oos.cummax()
#     max_dd_strat = ((equity_curve_oos - peak_strat) / peak_strat).min() if len(equity_curve_oos) > 0 else 0
#     total_trades_oos = pf_oos.trades.count()
    
#     close_oos_bh = close.loc[date_oos]
#     rets_bh = close_oos_bh.pct_change().dropna()
#     equity_bh = (1 + rets_bh).cumprod()
    
#     cagr_bh = (equity_bh.iloc[-1] / 1.0) ** (1 / anni_totali_oos) - 1 if len(equity_bh) > 0 and anni_totali_oos > 0 else 0
#     vol_bh = rets_bh.std() * np.sqrt(giorni_anno)
#     sharpe_bh = (rets_bh.mean() / rets_bh.std() * np.sqrt(giorni_anno)) if vol_bh > 0 else 0
    
#     peak_bh = equity_bh.cummax()
#     max_dd_bh = ((equity_bh - peak_bh) / peak_bh).min() if len(equity_bh) > 0 else 0
    
#     alpha = cagr_strat - cagr_bh
    
#     return {
#         "Param (F-S-SIG)": f"{best_f}-{best_s}-{best_sig}",
#         "Start_OOS": data_inizio_oos,
#         "Shp_IS": sharpe_is_scelto,
#         "Shp_IS_B&H": sharpe_bh_is,
#         "Shp_OOS": sharpe_strat,
#         "Shp_OOS_B&H": sharpe_bh,
#         "CAGR_Str": cagr_strat * 100,
#         "CAGR_B&H": cagr_bh * 100,
#         "Alpha_%": alpha * 100,
#         "DD_Str": max_dd_strat * 100,
#         "DD_B&H": max_dd_bh * 100,
#         "TRD_OOS": total_trades_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"⌛ Split 60-40 MACD per {ticker:<10} | Cluster: {cluster_appartenenza:<12} ... ", end="")
    
#     metrics = ottimizza_asset_split_60_40_macd(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#         print(f"COMPLETATO ✅ (MACD: {metrics['Param (F-S-SIG)']})")
#     elif metrics == "LOW_SHARPE":
#         print("SCARTATO (Nessun setup valido In-Sample) ⚠️")
#     elif metrics == "NO_STABLE_PLATEAU":
#         print("SCARTATO (Tutti i picchi erano isolati / overfittati) ⚠️")
#     else:
#         print("ERRORE/STORICO INSUFFICIENTE ❌")

# # ==============================================================================
# # 5. LEADERBOARD E STAMPA TABELLONE FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_%", ascending=False).reset_index(drop=True)

#     print("\n" + "="*155)
#     print("🏆 TABELLONE FINALE MACD - NEIGHBORHOOD SENSITIVITY (Ordinato per Alpha OOS)")
#     print("="*155)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Start_OOS", "Param (F-S-SIG)", "Shp_IS", "Shp_IS_B&H", "Shp_OOS", "Shp_OOS_B&H", "CAGR_Str", "CAGR_B&H", "Alpha_%", "DD_Str", "DD_B&H", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "F-S-SIG", "SHP IS", "SHP IS B&H", "SHP OOS", "SHP OOS B&H", "CAGR STR", "CAGR B&H", "ALPHA %", "DD STR", "DD B&H", "TRD"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':       '{:,.2f}'.format,
#             'SHP IS B&H':   '{:,.2f}'.format,
#             'SHP OOS':      '{:,.2f}'.format,
#             'SHP OOS B&H':  '{:,.2f}'.format,
#             'CAGR STR':     '{:,.1f}%'.format,
#             'CAGR B&H':     '{:,.1f}%'.format,
#             'ALPHA %':      '{:,.1f}%'.format,
#             'DD STR':       '{:,.1f}%'.format,
#             'DD B&H':       '{:,.1f}%'.format,
#             'TRD':          '{:,}'.format
#         }
#     ))
#     print("="*155)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore Split 60/40 MACD (Neighborhood Sensitivity Analysis) su 48 asset.
📦 Griglia tridimensionale generata: 14848 combinazioni valide.
⌛ Split 60-40 MACD per US100.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (MACD: 10-20-11)
⌛ Split 60-40 MACD per US30.cash  | Cluster: cash_cfd     ... COMPLETATO ✅ (MACD: 6-40-8)
⌛ Split 60-40 MACD per US500.cash | Cluster: cash_cfd     ... COMPLETATO ✅ (MACD: 10-24-14)
⌛ Split 60-40 MACD per BNBUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (MACD: 38-44-8)
⌛ Split 60-40 MACD per BTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (MACD: 48-56-32)
⌛ Split 60-40 MACD per ETHUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (MACD: 44-56-20)
⌛ Split 60-40 MACD per GRTUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (MACD: 34-40-68)
⌛ Split 60-40 MACD per LTCUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (MACD: 34-36-17)
⌛ Split 60-40 MACD per MANUSD     | Cluster: crypto_cfd   ... COMPLETATO ✅ (MACD: 50-52-5)
⌛ Split 60-40 MACD per NERUSD     | Cl

# conv 3d + dist. min. + ensemble

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# import gc  # Memory Management

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore MACD Split 60/40 (Convoluzione 3D + Distanza Ridotta + Ensemble) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE DELLA GRIGLIA PARAMETRICA TRIDIMENSIONALE (MACD)
# # ==============================================================================
# fast_range = np.arange(6, 68, 2)     # Fast EMA
# slow_range = np.arange(20, 100, 4)   # Slow EMA
# sig_range  = np.arange(5, 100, 3)    # Signal Line EMA

# griglia_parametri = [
#     (f, s, sig) 
#     for f in fast_range for s in slow_range for sig in sig_range 
#     if f < s  # La lenta deve essere maggiore della veloce
# ]
# print(f"📦 Griglia tridimensionale MACD generata: {len(griglia_parametri)} combinazioni valide.")

# # ==============================================================================
# # 3. CORE FUNCTION: SPLIT 60/40 + CONVOLUZIONE + ENSEMBLE ENGINE
# # ==============================================================================

# MIN_VICINI_VALIDI = 6         

# def calc_advanced_metrics(rets, giorni_anno):
#     """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
#     if len(rets) == 0 or rets.isnull().all():
#         return 0, 0, 0, 0, 0
        
#     # CAGR
#     anni = len(rets) / giorni_anno
#     cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
#     # Volatilità e Sharpe (Annualizzati)
#     daily_std = rets.std()
#     vol = daily_std * np.sqrt(giorni_anno)
#     sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
#     # Sortino (Annualizzato)
#     neg_rets = rets[rets < 0]
#     daily_down_std = neg_rets.std()
#     sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
#     # Max DD
#     cum = (1 + rets).cumprod()
#     peak = cum.cummax()
#     dd = (cum - peak) / peak
#     max_dd = dd.min() if len(dd) > 0 else 0
    
#     return cagr, vol, sharpe, sortino, max_dd

# def ottimizza_split_60_40_macd_ensemble(ticker, df, griglia, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     f_arr = [t[0] for t in griglia]
#     s_arr = [t[1] for t in griglia]
#     sig_arr = [t[2] for t in griglia]
#     multi_idx = pd.MultiIndex.from_tuples(griglia, names=['Fast', 'Slow', 'Signal'])
    
#     # Calcolo globale vettoriale del MACD
#     macd_global = vbt.MACD.run(close, fast_window=f_arr, slow_window=s_arr, signal_window=sig_arr)
    
#     macd_line = macd_global.macd
#     signal_line = macd_global.signal
    
#     macd_line.columns = multi_idx
#     signal_line.columns = multi_idx
    
#     entries_raw = macd_line.vbt.crossed_above(signal_line)
#     exits_raw = macd_line.vbt.crossed_below(signal_line)
    
#     entries_global = entries_raw.vbt.signals.fshift(1)
#     exits_global = exits_raw.vbt.signals.fshift(1)
    
#     # Liberiamo la RAM subito
#     del macd_global, macd_line, signal_line, entries_raw, exits_raw
#     gc.collect()
    
#     # DATI IS e OOS
#     close_is = close.iloc[:split_idx]
#     close_oos = close.iloc[split_idx:]
#     data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    
#     # --- CALCOLO BUY & HOLD IS/OOS ---
#     rets_bh_is = close_is.pct_change().dropna()
#     cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
#     rets_bh_oos = close_oos.pct_change().dropna()
#     cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)
    
#     # --- IN-SAMPLE STRATEGY CALCULATION ---
#     pf_is_all = vbt.Portfolio.from_signals(close_is, entries_global.iloc[:split_idx], exits_global.iloc[:split_idx], freq=None)
#     rets_is_all = pf_is_all.returns()
#     std_is_all = rets_is_all.std()
    
#     anni_is = len(close_is) / giorni_anno
#     tpy_is_array = pf_is_all.trades.count().values / anni_is if anni_is > 0 else 0
    
#     # RICHIEDIAMO ALMENO 2 TRADE ANNUI NELL'IN-SAMPLE
#     sharpe_array_is = np.where(
#         (std_is_all > 0) & (tpy_is_array >= 2), 
#         (rets_is_all.mean() / std_is_all) * np.sqrt(giorni_anno), 
#         np.nan
#     )
    
#     top_stable_params = []
    
#     if not np.isnan(sharpe_array_is).all():
#         shape_3d = (len(fast_range), len(slow_range), len(sig_range))
#         cubo_sharpe = np.full(shape_3d, np.nan)
#         for idx, (f, s, sig) in enumerate(griglia):
#             fi = np.where(fast_range == f)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             sigi = np.where(sig_range == sig)[0][0]
#             cubo_sharpe[fi, si, sigi] = sharpe_array_is[idx]

#         valid_indices = np.where(~np.isnan(sharpe_array_is))[0]
        
#         convolved_candidates = []
        
#         # 🔥 FASE DI CONVOLUZIONE 3D 🔥
#         for candidate_idx in valid_indices:
#             f, s, sig = griglia[candidate_idx]
            
#             fi = np.where(fast_range == f)[0][0]
#             si = np.where(slow_range == s)[0][0]
#             sigi = np.where(sig_range == sig)[0][0]
            
#             # Estrazione del cubo 3x3x3 (Vicinato + Cella Centrale)
#             fi_min, fi_max = max(0, fi-1), min(shape_3d[0], fi+2)
#             si_min, si_max = max(0, si-1), min(shape_3d[1], si+2)
#             sigi_min, sigi_max = max(0, sigi-1), min(shape_3d[2], sigi+2)
            
#             neighborhood = cubo_sharpe[fi_min:fi_max, si_min:si_max, sigi_min:sigi_max].flatten()
#             valid_neighbors = neighborhood[~np.isnan(neighborhood)]
            
#             if len(valid_neighbors) >= MIN_VICINI_VALIDI:
#                 # Score Convoluzione = Media(Sharpe) - DeviazioneStandard(Sharpe)
#                 conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
#                 convolved_candidates.append((conv_score, (f, s, sig)))
                
#         # Ordiniamo tutti i candidati in base al loro Score di Convoluzione
#         convolved_candidates.sort(key=lambda x: x[0], reverse=True)
        
#         # 🔥 FIX: DISTANZIAMENTO SPAZIALE RIDOTTO (SENZA PALI DELLA PORTA) 🔥
#         DISTANZA_MINIMA = 10 # Distanza ridotta per permettere varianti sulla stessa frequenza
        
#         for score, params in convolved_candidates:
#             f, s, sig = params
#             troppo_vicino = False
            
#             # Controlliamo solo la distanza totale combinata
#             for (scelto_f, scelto_s, scelto_sig) in top_stable_params:
#                 distanza_totale = abs(f - scelto_f) + abs(s - scelto_s) + abs(sig - scelto_sig)
                
#                 # Regola: Distanza matematica minima
#                 if distanza_totale < DISTANZA_MINIMA:
#                     troppo_vicino = True
#                     break 
            
#             # Se supera il controllo, lo aggiungiamo all'Ensemble
#             if not troppo_vicino:
#                 top_stable_params.append((f, s, sig))
#                 if len(top_stable_params) == 3: # Ci fermiamo a 3
#                     break

#     del pf_is_all, rets_is_all, std_is_all
#     if 'cubo_sharpe' in locals(): del cubo_sharpe
#     gc.collect()

#     # --- ENSEMBLE CALCULATION ---
#     if not top_stable_params:
#         return "LOW_SHARPE"
        
#     is_rets_list, oos_rets_list = [], []
#     is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
#     for p_tuple in top_stable_params:
#         # IS singolo parametro
#         pf_i = vbt.Portfolio.from_signals(close_is, entries_global[p_tuple].iloc[:split_idx], exits_global[p_tuple].iloc[:split_idx], freq=None)
#         is_rets_list.append(pf_i.returns())
#         is_trd.append(pf_i.trades.count())
#         is_win.append((pf_i.trades.pnl > 0).sum())
        
#         # OOS singolo parametro
#         pf_o = vbt.Portfolio.from_signals(close_oos, entries_global[p_tuple].iloc[split_idx:], exits_global[p_tuple].iloc[split_idx:], freq=None)
#         oos_rets_list.append(pf_o.returns())
#         oos_trd.append(pf_o.trades.count())
#         oos_win.append((pf_o.trades.pnl > 0).sum())
        
#     # Blending (Media equiponderata al 33%)
#     blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
#     blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
#     # Metriche Ensemble IS
#     cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
#     avg_trd_is = np.mean(is_trd)
#     win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
#     # Metriche Ensemble OOS
#     cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
#     avg_trd_oos = np.mean(oos_trd)
#     win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
#     alpha_oos = cagr_oos - cagr_bh_oos
#     param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
#     # Pulizia memoria finale asset
#     del entries_global, exits_global
#     gc.collect()

#     return {
#         "Parametri": param_str,
#         "Inizio_OOS": data_inizio_oos,
#         "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
#         "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
#         "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
#         "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
#         "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
#         "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
#         "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
#         "Alpha_OOS": alpha_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"\n============================================================")
#     print(f"⌛ ELABORAZIONE MACD {ticker:<8} | Cluster: {cluster_appartenenza}")
    
#     metrics = ottimizza_split_60_40_macd_ensemble(ticker, df, griglia_parametri, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         # Stampa a video dettagliata per singolo ticker
#         print(f"✅ Parametri: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
#         print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
#         print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
#         print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
#         print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
#         print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#     elif metrics == "LOW_SHARPE":
#         print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
#     else:
#         print("❌ ERRORE: Storico dati insufficiente.")

# # ==============================================================================
# # 5. LEADERBOARD FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

#     print("\n" + "="*160)
#     print("🏆 TABELLONE FINALE MACD ENSEMBLE (Ordinato per Alpha OOS)")
#     print("="*160)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
#         "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
#         "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (F-S-Sig)", 
#         "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
#         "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'B&H IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'B&H OOS':   '{:,.2f}'.format,
#             'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
#             'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
#             'ALPHA %':   lambda x: f"{x*100:,.1f}%",
#             'DD OOS':    lambda x: f"{x*100:,.1f}%",
#             'B&H DD':    lambda x: f"{x*100:,.1f}%",
#             'TRD OOS':   '{:,.1f}'.format
#         }
#     ))
#     print("="*160)

🚀 Motore MACD Split 60/40 (Convoluzione 3D + Distanza Ridotta + Ensemble) su 48 asset.
📦 Griglia tridimensionale MACD generata: 14848 combinazioni valide.

⌛ ELABORAZIONE MACD US100.cash | Cluster: cash_cfd
✅ Parametri: [(10,20,11), (56,60,14), (54,56,8)] | Inizio OOS: 2024-04-09
  [IS]  STRAT -> Shp: 1.21 | Cagr: 14.7% | DD: -14.5% | Sort: 1.68 | Trd: 39.0 | Win: 58.1%
  [IS]  B&H   -> Shp: 0.51 | Cagr: 9.5% | DD: -35.6% | Sort: 0.74
  [OOS] STRAT -> Shp: 0.73 | Cagr: 6.8% | DD: -13.1% | Sort: 0.74 | Trd: 31.3 | Win: 54.3%
  [OOS] B&H   -> Shp: 1.21 | Cagr: 25.9% | DD: -22.8% | Sort: 1.65
  🔥 ALPHA OOS:  -19.11%

⌛ ELABORAZIONE MACD US30.cash | Cluster: cash_cfd
✅ Parametri: [(6,40,11), (6,36,5), (6,52,5)] | Inizio OOS: 2023-06-28
  [IS]  STRAT -> Shp: 0.78 | Cagr: 8.4% | DD: -14.1% | Sort: 0.76 | Trd: 52.7 | Win: 60.1%
  [IS]  B&H   -> Shp: 0.42 | Cagr: 6.9% | DD: -37.1% | Sort: 0.50
  [OOS] STRAT -> Shp: 0.44 | Cagr: 3.0% | DD: -12.7% | Sort: 0.48 | Trd: 39.3 | Win: 49.2%
  [OOS] B&

# conv. 3d + ensemble + fast e slow ancorate

In [ ]:
# import os
# import json
# import numpy as np
# import pandas as pd
# import vectorbt as vbt
# import gc  # Memory Management

# # ==============================================================================
# # 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE
# # ==============================================================================
# path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
# with open(path_mappa, "r") as f:
#     mappa_strategie = json.load(f)

# ticker_target = mappa_strategie["tier_1_strong_trend"] + mappa_strategie["tier_2_mid_trend"]

# all_tickers_dict = {}
# for cluster, tickers in dati_puliti.items():
#     for t, df in tickers.items():
#         if t in ticker_target:
#             all_tickers_dict[t] = {"df": df, "cluster": cluster}

# print(f"🚀 Motore MACD Split 60/40 (Ancorato + Convoluzione 1D + Ensemble) su {len(all_tickers_dict)} asset.")

# # ==============================================================================
# # 2. GENERAZIONE RANGE PARAMETRI (MACD)
# # ==============================================================================
# fast_range = np.arange(4, 34, 2)     # Fast EMA
# slow_range = np.arange(16, 72, 4)   # Slow EMA
# sig_range  = np.arange(5, 50, 2)    # Signal Line EMA

# # ==============================================================================
# # 3. CORE FUNCTION: SPLIT 60/40 + MACD ANCORATO + ENSEMBLE ENGINE
# # ==============================================================================

# def calc_advanced_metrics(rets, giorni_anno):
#     """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
#     if len(rets) == 0 or rets.isnull().all():
#         return 0, 0, 0, 0, 0
        
#     # CAGR
#     anni = len(rets) / giorni_anno
#     cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
#     # Volatilità e Sharpe (Annualizzati)
#     daily_std = rets.std()
#     vol = daily_std * np.sqrt(giorni_anno)
#     sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
#     # Sortino (Annualizzato)
#     neg_rets = rets[rets < 0]
#     daily_down_std = neg_rets.std()
#     sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
#     # Max DD
#     cum = (1 + rets).cumprod()
#     peak = cum.cummax()
#     dd = (cum - peak) / peak
#     max_dd = dd.min() if len(dd) > 0 else 0
    
#     return cagr, vol, sharpe, sortino, max_dd


# def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
#     close = df['Close'].copy()
#     close.index = pd.to_datetime(close.index)
    
#     giorni_anno = 365 if is_crypto else 252
#     total_bars = len(close)
#     split_idx = int(total_bars * 0.6)
    
#     if split_idx < 200 or (total_bars - split_idx) < 50:
#         return None
        
#     # DATI IS e OOS
#     close_is = close.iloc[:split_idx]
#     close_oos = close.iloc[split_idx:]
#     data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
#     anni_is = len(close_is) / giorni_anno
    
#     # --- CALCOLO BUY & HOLD IS/OOS ---
#     rets_bh_is = close_is.pct_change().dropna()
#     cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
#     rets_bh_oos = close_oos.pct_change().dropna()
#     cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

#     # ==============================================================
#     # 1. TROVA IL TELAIO OTTIMO (Fast, Slow) In-Sample
#     # ==============================================================
#     # Usiamo l'approccio vettoriale di vectorbt per simulare il telaio in un istante
#     fs_combs = [(f, s) for f in fast_range for s in slow_range if f < s]
#     f_arr = [c[0] for c in fs_combs]
#     s_arr = [c[1] for c in fs_combs]
    
#     macd_frame = vbt.MACD.run(close_is, fast_window=f_arr, slow_window=s_arr, signal_window=9)
#     entries_frame = (macd_frame.macd > macd_frame.signal).vbt.signals.fshift(1)
#     exits_frame = (macd_frame.macd < macd_frame.signal).vbt.signals.fshift(1)
    
#     pf_frame = vbt.Portfolio.from_signals(close_is, entries_frame, exits_frame, freq='D')
    
#     std_frame = pf_frame.returns().std()
#     tpy_frame = pf_frame.trades.count() / anni_is if anni_is > 0 else 0
#     sharpes_frame = pf_frame.sharpe_ratio()
    
#     # Validiamo solo quelli con deviazione standard > 0 e almeno 2 trade all'anno
#     valid_sharpes = np.where((std_frame > 0) & (tpy_frame >= 2), sharpes_frame, np.nan)
    
#     if np.isnan(valid_sharpes).all():
#         del macd_frame, entries_frame, exits_frame, pf_frame
#         gc.collect()
#         return "LOW_SHARPE"
        
#     best_idx = np.nanargmax(valid_sharpes)
#     best_f = f_arr[best_idx]
#     best_s = s_arr[best_idx]
    
#     del macd_frame, entries_frame, exits_frame, pf_frame
#     gc.collect()
    
#     # ==============================================================
#     # 2. ENSEMBLE SULLE 3 SIGNAL LINE MIGLIORI PER IL TELAIO
#     # ==============================================================
#     macd_sig = vbt.MACD.run(close_is, fast_window=best_f, slow_window=best_s, signal_window=sig_range)
#     entries_sig = (macd_sig.macd > macd_sig.signal).vbt.signals.fshift(1)
#     exits_sig = (macd_sig.macd < macd_sig.signal).vbt.signals.fshift(1)
    
#     pf_sig = vbt.Portfolio.from_signals(close_is, entries_sig, exits_sig, freq='D')
    
#     std_sig = pf_sig.returns().std()
#     tpy_sig = pf_sig.trades.count() / anni_is if anni_is > 0 else 0
#     sharpes_sig = pf_sig.sharpe_ratio()
    
#     valid_sig_sharpes = np.where((std_sig > 0) & (tpy_sig >= 2), sharpes_sig, np.nan)
    
#     convolved_candidates = []
    
#     # 🔥 FASE DI CONVOLUZIONE 1D SULLA SIGNAL LINE 🔥
#     for i, sig in enumerate(sig_range):
#         if np.isnan(valid_sig_sharpes[i]):
#             continue
            
#         # Estraiamo il vicinato 1D (Signal Line precedente e successiva)
#         idx_min = max(0, i-1)
#         idx_max = min(len(sig_range), i+2)
#         vicinato = valid_sig_sharpes[idx_min:idx_max]
#         valid_neighbors = vicinato[~np.isnan(vicinato)]
        
#         if len(valid_neighbors) >= 2: # Minimo 2 elementi per considerarlo un altopiano
#             # Score = Media(Sharpe) - DeviazioneStandard(Sharpe)
#             conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
#             convolved_candidates.append((conv_score, sig))
            
#     convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
#     # 🔥 DISTANZIAMENTO 1D SULLA SIGNAL LINE 🔥
#     DISTANZA_MINIMA = 10 
#     top_stable_sigs = []
    
#     for score, sig in convolved_candidates:
#         # Controlliamo che la Signal line sia distante almeno 10 periodi da quelle già scelte
#         if all(abs(sig - t) >= DISTANZA_MINIMA for t in top_stable_sigs):
#             top_stable_sigs.append(sig)
#         if len(top_stable_sigs) == 3: 
#             break
            
#     if not top_stable_sigs:
#         del macd_sig, entries_sig, exits_sig, pf_sig
#         gc.collect()
#         return "LOW_SHARPE"
        
#     del macd_sig, entries_sig, exits_sig, pf_sig
#     gc.collect()

#     # ==============================================================
#     # 3. ENSEMBLE CALCULATION (IS e OOS)
#     # ==============================================================
#     macd_final = vbt.MACD.run(close, fast_window=best_f, slow_window=best_s, signal_window=top_stable_sigs)
#     entries_final = (macd_final.macd > macd_final.signal).vbt.signals.fshift(1)
#     exits_final = (macd_final.macd < macd_final.signal).vbt.signals.fshift(1)
    
#     is_rets_list, oos_rets_list = [], []
#     is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
#     for sig in top_stable_sigs:
#         idx_tuple = (best_f, best_s, sig)
        
#         # IS singolo parametro
#         pf_i = vbt.Portfolio.from_signals(close_is, entries_final[idx_tuple].iloc[:split_idx], exits_final[idx_tuple].iloc[:split_idx], freq='D')
#         is_rets_list.append(pf_i.returns())
#         is_trd.append(pf_i.trades.count())
#         is_win.append((pf_i.trades.pnl > 0).sum())
        
#         # OOS singolo parametro
#         pf_o = vbt.Portfolio.from_signals(close_oos, entries_final[idx_tuple].iloc[split_idx:], exits_final[idx_tuple].iloc[split_idx:], freq='D')
#         oos_rets_list.append(pf_o.returns())
#         oos_trd.append(pf_o.trades.count())
#         oos_win.append((pf_o.trades.pnl > 0).sum())
        
#     # Blending (Media equiponderata al 33%)
#     blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
#     blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
#     # Metriche Ensemble IS
#     cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
#     avg_trd_is = np.mean(is_trd)
#     win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
#     # Metriche Ensemble OOS
#     cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
#     avg_trd_oos = np.mean(oos_trd)
#     win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
#     alpha_oos = cagr_oos - cagr_bh_oos
    
#     # Formattazione visiva ancorata: (Fast, Slow, [Sig1, Sig2, Sig3])
#     param_str = f"({int(best_f)},{int(best_s)},[{','.join(map(str, [int(s) for s in top_stable_sigs]))}])"
    
#     # Pulizia memoria finale asset
#     del macd_final, entries_final, exits_final
#     gc.collect()

#     return {
#         "Parametri": param_str,
#         "Inizio_OOS": data_inizio_oos,
#         "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
#         "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
#         "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
#         "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
#         "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
#         "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
#         "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
#         "Alpha_OOS": alpha_oos
#     }

# # ==============================================================================
# # 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# # ==============================================================================
# report_finale = []

# for ticker, info in all_tickers_dict.items():
#     df = info["df"]
#     cluster_appartenenza = info["cluster"]
#     is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
#     print(f"\n============================================================")
#     print(f"⌛ ELABORAZIONE MACD {ticker:<8} | Cluster: {cluster_appartenenza}")
    
#     metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
#     if isinstance(metrics, dict):
#         # Stampa a video dettagliata per singolo ticker
#         print(f"✅ Parametri: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
#         print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
#         print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
#         print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
#         print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
#         print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
#         metrics["Ticker"] = ticker
#         metrics["Cluster"] = cluster_appartenenza
#         report_finale.append(metrics)
#     elif metrics == "LOW_SHARPE":
#         print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
#     else:
#         print("❌ ERRORE: Storico dati insufficiente.")

# # ==============================================================================
# # 5. LEADERBOARD FINALE
# # ==============================================================================
# if len(report_finale) > 0:
#     df_leaderboard = pd.DataFrame(report_finale)
#     df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

#     print("\n" + "="*160)
#     print("🏆 TABELLONE FINALE MACD ENSEMBLE ANCORATO (Ordinato per Alpha OOS)")
#     print("="*160)

#     tabella_stampa = df_leaderboard[[
#         "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
#         "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
#         "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
#     ]].copy()

#     tabella_stampa.columns = [
#         "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (F-S-[Sigs])", 
#         "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
#         "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
#     ]

#     print(tabella_stampa.to_string(
#         index=False, 
#         justify='left',
#         col_space=9,
#         formatters={
#             'SHP IS':    '{:,.2f}'.format,
#             'B&H IS':    '{:,.2f}'.format,
#             'SHP OOS':   '{:,.2f}'.format,
#             'B&H OOS':   '{:,.2f}'.format,
#             'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
#             'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
#             'ALPHA %':   lambda x: f"{x*100:,.1f}%",
#             'DD OOS':    lambda x: f"{x*100:,.1f}%",
#             'B&H DD':    lambda x: f"{x*100:,.1f}%",
#             'TRD OOS':   '{:,.1f}'.format
#         }
#     ))
#     print("="*160)
# else:
#     print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore MACD Split 60/40 (Ancorato + Convoluzione 1D + Ensemble) su 48 asset.

⌛ ELABORAZIONE MACD US100.cash | Cluster: cash_cfd
✅ Parametri: (10,24,[5,15,37]) | Inizio OOS: 2024-04-09
  [IS]  STRAT -> Shp: 0.96 | Cagr: 13.0% | DD: -14.6% | Sort: 1.17 | Trd: 29.3 | Win: 54.5%
  [IS]  B&H   -> Shp: 0.51 | Cagr: 9.5% | DD: -35.6% | Sort: 0.74
  [OOS] STRAT -> Shp: 0.88 | Cagr: 10.1% | DD: -17.0% | Sort: 0.81 | Trd: 22.3 | Win: 49.3%
  [OOS] B&H   -> Shp: 1.21 | Cagr: 25.9% | DD: -22.8% | Sort: 1.65
  🔥 ALPHA OOS:  -15.79%

⌛ ELABORAZIONE MACD US30.cash | Cluster: cash_cfd
✅ Parametri: (18,32,[9,19,39]) | Inizio OOS: 2023-06-28
  [IS]  STRAT -> Shp: 0.70 | Cagr: 6.7% | DD: -16.9% | Sort: 0.67 | Trd: 26.3 | Win: 63.3%
  [IS]  B&H   -> Shp: 0.42 | Cagr: 6.9% | DD: -37.1% | Sort: 0.50
  [OOS] STRAT -> Shp: 0.55 | Cagr: 4.3% | DD: -8.5% | Sort: 0.62 | Trd: 21.0 | Win: 55.6%
  [OOS] B&H   -> Shp: 1.06 | Cagr: 14.4% | DD: -16.6% | Sort: 1.52
  🔥 ALPHA OOS:  -10.08%

⌛ ELABORAZIONE MACD US500.

# conv. 2d e 1d + ens

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CLONATO DA 3EMA)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Struttura originale identica alla 3EMA per garantire lo stesso identico conteggio (90 asset)
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# Protection Lock: Se la RAM è vuota recupera direttamente dai parquet locali
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 2D + 1D Ancorata + Ensemble) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI (MACD)
# ==============================================================================
fast_range = np.arange(4, 34, 2)     # Fast EMA
slow_range = np.arange(16, 72, 4)    # Slow EMA
sig_range  = np.arange(5, 50, 2)     # Signal Line EMA

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 ASINCRONO + MACD ANCORATO + ENSEMBLE ENGINE
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
        
    # CAGR
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
    # Volatilità e Sharpe (Annualizzati)
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
    # Sortino (Annualizzato)
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
    # Max DD
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
    # 1. SALVATAGGIO CRONOLOGIA REALE
    date_vere = df.index.copy()
    
    close = df['Close'].copy()
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    # Estraiamo la data reale di inizio OOS dai metadati
    data_inizio_oos = pd.to_datetime(date_vere[split_idx]).strftime('%Y-%m-%d')
    giorni_anno = 365 if is_crypto else 252
    
    # Resettiamo l'indice a numeri puri (0, 1, 2...). VectorBT lavorerà in isolamento sequenziale
    close.index = np.arange(total_bars)
    
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. TROVA IL TELAIO OTTIMO (Fast, Slow) In-Sample (CONVOLUZIONE 2D)
    # ==============================================================
    fs_combs = [(f, s) for f in fast_range for s in slow_range if f < s]
    f_arr = [c[0] for c in fs_combs]
    s_arr = [c[1] for c in fs_combs]
    
    macd_frame = vbt.MACD.run(close, fast_window=f_arr, slow_window=s_arr, signal_window=9)
    entries_frame_raw = (macd_frame.macd > macd_frame.signal)
    exits_frame_raw = (macd_frame.macd < macd_frame.signal)
    
    entries_frame_global = entries_frame_raw.vbt.signals.fshift(1)
    exits_frame_global = exits_frame_raw.vbt.signals.fshift(1)
    
    pf_frame = vbt.Portfolio.from_signals(
        close_is, 
        entries_frame_global.iloc[:split_idx], 
        exits_frame_global.iloc[:split_idx], 
        freq=None
    )
    
    # 🔥 CALCOLO SHARPE MANUALE IDENTICO ALLA 3EMA (Evita il ValueError)
    rets_frame = pf_frame.returns()
    std_frame = rets_frame.std()
    tpy_frame = pf_frame.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_frame = (rets_frame.mean() / std_frame) * np.sqrt(giorni_anno)
    
    valid_sharpes = np.where((std_frame > 0) & (tpy_frame >= 2), sharpes_frame, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del macd_frame, entries_frame_global, exits_frame_global, pf_frame
        gc.collect()
        return "LOW_SHARPE"
        
    shape_2d = (len(fast_range), len(slow_range))
    mat_sharpe = np.full(shape_2d, np.nan)
    
    for idx, (f, s) in enumerate(fs_combs):
        f_idx = np.where(fast_range == f)[0][0]
        s_idx = np.where(slow_range == s)[0][0]
        mat_sharpe[f_idx, s_idx] = valid_sharpes[idx]
        
    best_conv_score_2d = -np.inf
    best_fs = None
    
    for f_idx, f in enumerate(fast_range):
        for s_idx, s in enumerate(slow_range):
            if np.isnan(mat_sharpe[f_idx, s_idx]):
                continue
                
            f_min, f_max = max(0, f_idx-1), min(shape_2d[0], f_idx+2)
            s_min, s_max = max(0, s_idx-1), min(shape_2d[1], s_idx+2)
            
            vicinato = mat_sharpe[f_min:f_max, s_min:s_max].flatten()
            valid_neighbors = vicinato[~np.isnan(vicinato)]
            
            if len(valid_neighbors) >= 4: 
                conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                if conv_score > best_conv_score_2d:
                    best_conv_score_2d = conv_score
                    best_fs = (f, s)
                    
    if best_fs is None:
        del macd_frame, entries_frame_global, exits_frame_global, pf_frame
        gc.collect()
        return "LOW_SHARPE"
        
    best_f, best_s = best_fs
    
    del macd_frame, entries_frame_global, exits_frame_global, pf_frame
    gc.collect()
    
    # ==============================================================
    # 2. ENSEMBLE SULLE 3 SIGNAL LINE MIGLIORI (CONVOLUZIONE 1D)
    # ==============================================================
    macd_sig = vbt.MACD.run(close, fast_window=best_f, slow_window=best_s, signal_window=sig_range)
    entries_sig_raw = (macd_sig.macd > macd_sig.signal)
    exits_sig_raw = (macd_sig.macd < macd_sig.signal)
    
    entries_sig_global = entries_sig_raw.vbt.signals.fshift(1)
    exits_sig_global = exits_sig_raw.vbt.signals.fshift(1)
    
    pf_sig = vbt.Portfolio.from_signals(
        close_is, 
        entries_sig_global.iloc[:split_idx], 
        exits_sig_global.iloc[:split_idx], 
        freq=None
    )
    
    # 🔥 CALCOLO SHARPE MANUALE IDENTICO ALLA 3EMA
    rets_sig = pf_sig.returns()
    std_sig = rets_sig.std()
    tpy_sig = pf_sig.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_sig = (rets_sig.mean() / std_sig) * np.sqrt(giorni_anno)
    
    valid_sig_sharpes = np.where((std_sig > 0) & (tpy_sig >= 2), sharpes_sig, np.nan)
    convolved_candidates = []
    
    for i, sig in enumerate(sig_range):
        if np.isnan(valid_sig_sharpes[i]):
            continue
            
        idx_min = max(0, i-1)
        idx_max = min(len(sig_range), i+2)
        vicinato = valid_sig_sharpes[idx_min:idx_max]
        valid_neighbors = vicinato[~np.isnan(vicinato)]
        
        if len(valid_neighbors) >= 2: 
            conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
            convolved_candidates.append((conv_score, sig))
            
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    DISTANZA_MINIMA = 10 
    top_stable_sigs = []
    
    for score, sig in convolved_candidates:
        if all(abs(sig - t) >= DISTANZA_MINIMA for t in top_stable_sigs):
            top_stable_sigs.append(sig)
        if len(top_stable_sigs) == 3: 
            break
            
    if not top_stable_sigs:
        del macd_sig, entries_sig_global, exits_sig_global, pf_sig
        gc.collect()
        return "LOW_SHARPE"
        
    del macd_sig, entries_sig_global, exits_sig_global, pf_sig
    gc.collect()

    # ==============================================================
    # 3. ENSEMBLE CALCULATION (IS e OOS)
    # ==============================================================
    macd_final = vbt.MACD.run(close, fast_window=best_f, slow_window=best_s, signal_window=top_stable_sigs)
    entries_final_raw = (macd_final.macd > macd_final.signal)
    exits_final_raw = (macd_final.macd < macd_final.signal)
    
    entries_final_global = entries_final_raw.vbt.signals.fshift(1)
    exits_final_global = exits_final_raw.vbt.signals.fshift(1)
    
    is_rets_list, oos_rets_list = [], []
    is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
    for sig in top_stable_sigs:
        idx_tuple = (best_f, best_s, sig)
        
        pf_i = vbt.Portfolio.from_signals(close_is, entries_final_global[idx_tuple].iloc[:split_idx], exits_final_global[idx_tuple].iloc[:split_idx], freq=None)
        is_rets_list.append(pf_i.returns())
        is_trd.append(pf_i.trades.count())
        is_win.append((pf_i.trades.pnl > 0).sum())
        
        pf_o = vbt.Portfolio.from_signals(close_oos, entries_final_global[idx_tuple].iloc[split_idx:], exits_final_global[idx_tuple].iloc[split_idx:], freq=None)
        oos_rets_list.append(pf_o.returns())
        oos_trd.append(pf_o.trades.count())
        oos_win.append((pf_o.trades.pnl > 0).sum())
        
    blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
    blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
    avg_trd_is = np.mean(is_trd)
    win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
    avg_trd_oos = np.mean(oos_trd)
    win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = f"({int(best_f)},{int(best_s)},[{','.join(map(str, [int(s) for s in top_stable_sigs]))}])"
    
    del macd_final, entries_final_global, exits_final_global
    gc.collect()

    return {
        "Parametri": param_str,
        "Inizio_OOS": data_inizio_oos,
        "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
        "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
        "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
        "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
        "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
        "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE MACD {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE MACD ENSEMBLE ANCORATO (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (F-S-[Sigs])", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

🚀 Motore MACD Split 60/40 (Convoluzione 2D + 1D Ancorata + Ensemble) su 62 asset.

⌛ ELABORAZIONE MACD US100.cash | Cluster: cash_cfd


ValueError: Index frequency is None. Pass it as `freq` or define it globally under `settings.array_wrapper`.

# conv 3d + 3 ens 

In [2]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CON AUTO-RECOVER DA PARQUET)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Ricerca flessibile per evitare KeyError se i nomi dei Tier nel JSON sono estesi
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# --- PROTEZIONE DA NAMEERROR: SE LA RAM È VUOTA, LEGGE DIRETTAMENTE I PARQUET TARGET ---
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 3D Pura + Distanziamento >= 15) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI (MACD)
# ==============================================================================
fast_range = np.arange(4, 68, 2)     # Fast EMA
slow_range = np.arange(16, 100, 4)    # Slow EMA
sig_range  = np.arange(5, 100, 5)     # Signal Line EMA

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 ASINCRONO + MACD CONVOLUZIONE 3D + ENSEMBLE ENGINE
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
        
    # CAGR
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
    # Volatilità e Sharpe (Annualizzati)
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
    # Sortino (Annualizzato)
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
    # Max DD
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
    # Scudo contro i prezzi a 0.0 provenienti dai bad-tick di MT5
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    close.index = pd.to_datetime(close.index)
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. GENERAZIONE COMBINAZIONI TRIDIMENSIONALI E VETTORIZZAZIONE IS
    # ==============================================================
    griglia_3d = [(f, s, sig) for f in fast_range for s in slow_range if f < s for sig in sig_range]
    f_arr = [c[0] for c in griglia_3d]
    s_arr = [c[1] for c in griglia_3d]
    sig_arr = [c[2] for c in griglia_3d]
    
    macd_is = vbt.MACD.run(close_is, fast_window=f_arr, slow_window=s_arr, signal_window=sig_arr, param_product=False)
    
    # Generazione dei Crossover impulsivi (Stile 3EMA) per proteggere freq='D'
    entries_is = macd_is.macd.vbt.crossed_above(macd_is.signal).vbt.signals.fshift(1)
    exits_is = macd_is.macd.vbt.crossed_below(macd_is.signal).vbt.signals.fshift(1)
    
    pf_is = vbt.Portfolio.from_signals(close_is, entries_is, exits_is, freq='D')
    
    std_is = pf_is.returns().std()
    tpy_is = pf_is.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_is = pf_is.sharpe_ratio()
    
    # Validiamo solo canali stabili con trade sufficienti
    valid_sharpes = np.where((std_is > 0) & (tpy_is >= 2), sharpes_is, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del macd_is, entries_is, exits_is, pf_is
        gc.collect()
        return "LOW_SHARPE"
        
    # ==============================================================
    # 2. COSTRUZIONE CUBO E FASE DI CONVOLUZIONE 3D PURA
    # ==============================================================
    shape_3d = (len(fast_range), len(slow_range), len(sig_range))
    cubo_sharpe = np.full(shape_3d, np.nan)
    
    for idx, (f, s, sig) in enumerate(griglia_3d):
        f_idx = np.where(fast_range == f)[0][0]
        s_idx = np.where(slow_range == s)[0][0]
        sig_idx = np.where(sig_range == sig)[0][0]
        cubo_sharpe[f_idx, s_idx, sig_idx] = valid_sharpes[idx]
        
    convolved_candidates = []
    
    # Scansione tridimensionale dello spazio dei parametri
    for f_idx in range(shape_3d[0]):
        for s_idx in range(shape_3d[1]):
            for sig_idx in range(shape_3d[2]):
                if np.isnan(cubo_sharpe[f_idx, s_idx, sig_idx]):
                    continue
                    
                # Limiti del vicinato cubico 3x3x3 (27 celle complessive)
                f_min, f_max = max(0, f_idx-1), min(shape_3d[0], f_idx+2)
                s_min, s_max = max(0, s_idx-1), min(shape_3d[1], s_idx+2)
                sig_min, sig_max = max(0, sig_idx-1), min(shape_3d[2], sig_idx+2)
                
                vicinato = cubo_sharpe[f_min:f_max, s_min:s_max, sig_min:sig_max].flatten()
                valid_neighbors = vicinato[~np.isnan(vicinato)]
                
                # Richiediamo un altopiano solido: almeno 10 vicini validi su 27 nel cubo
                if len(valid_neighbors) >= 10:
                    conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                    convolved_candidates.append((conv_score, (fast_range[f_idx], slow_range[s_idx], sig_range[sig_idx])))
                    
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    # ==============================================================
    # 3. DISTANZIAMENTO SPAZIALE 3D (MANHATTAN DISTANCE >= 15)
    # ==============================================================
    DISTANZA_MINIMA = 15
    top_stable_params = []
    
    for score, p_triplet in convolved_candidates:
        f, s, sig = p_triplet
        troppo_vicino = False
        
        for (sf, ss, ssig) in top_stable_params:
            # Distanza geometrica combinata tra i tre assi parametrici
            distanza_totale = abs(f - sf) + abs(s - ss) + abs(sig - ssig)
            if  distanza_totale < DISTANZA_MINIMA:
                troppo_vicino = True
                break
                
        if not troppo_vicino:
            top_stable_params.append((f, s, sig))
            if len(top_stable_params) == 3:  # Ci fermiamo ai 3 picchi indipendenti migliori
                break
                
    del macd_is, entries_is, exits_is, pf_is
    gc.collect()
    
    if not top_stable_params:
        return "LOW_SHARPE"
        
    # ==============================================================
    # 4. CALCOLO ENSEMBLE FINALE CROSS-ASSET (1-TO-1 MAPPING)
    # ==============================================================
    f_top = [p[0] for p in top_stable_params]
    s_top = [p[1] for p in top_stable_params]
    sig_top = [p[2] for p in top_stable_params]
    
    # param_product=False accoppia gli indici in maniera lineare 1-a-1 generando esattamente 3 colonne
    macd_final = vbt.MACD.run(close, fast_window=f_top, slow_window=s_top, signal_window=sig_top, param_product=False)
    entries_final = macd_final.macd.vbt.crossed_above(macd_final.signal).vbt.signals.fshift(1)
    exits_final = macd_final.macd.vbt.crossed_below(macd_final.signal).vbt.signals.fshift(1)
    
    is_rets_list, oos_rets_list = [], []
    is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
    for idx in range(len(top_stable_params)):
        # Elaborazione posizionale ultra-sicura per evitare i conflitti di MultiIndex
        pf_i = vbt.Portfolio.from_signals(close_is, entries_final.iloc[:, idx].iloc[:split_idx], exits_final.iloc[:, idx].iloc[:split_idx], freq='D')
        is_rets_list.append(pf_i.returns())
        is_trd.append(pf_i.trades.count())
        is_win.append((pf_i.trades.pnl > 0).sum())
        
        pf_o = vbt.Portfolio.from_signals(close_oos, entries_final.iloc[:, idx].iloc[split_idx:], exits_final.iloc[:, idx].iloc[split_idx:], freq='D')
        oos_rets_list.append(pf_o.returns())
        oos_trd.append(pf_o.trades.count())
        oos_win.append((pf_o.trades.pnl > 0).sum())
        
    blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
    blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
    avg_trd_is = np.mean(is_trd)
    win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
    avg_trd_oos = np.mean(oos_trd)
    win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
    del macd_final, entries_final, exits_final
    gc.collect()

    return {
        "Parametri": param_str,
        "Inizio_OOS": data_inizio_oos,
        "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
        "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
        "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
        "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
        "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
        "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE MACD 3D {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE MACD ENSEMBLE 3D (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Ensemble 3D)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 3D Pura + Distanziamento >= 15) su 90 asset.

⌛ ELABORAZIONE MACD 3D COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: [(14,32,25), (16,40,20), (60,64,60)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 2.11 | Cagr: 95.1% | DD: -25.7% | Sort: 2.46 | Trd: 7.7 | Win: 73.9%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: -0.26 | Cagr: -13.8% | DD: -51.3% | Sort: -0.36 | Trd: 5.0 | Win: 33.3%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  40.24%

⌛ ELABORAZIONE MACD 3D COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: [(56,76,95), (62,72,85), (60,84,35)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.91 | Cagr: 48.2% | DD: -13.1% | Sort: 2.11 | Trd: 5.3 | Win: 87.5%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2.53
  [OOS] STRAT -> Shp: -0.04 | Cagr: 

# conv. 3d + slow param 1.7 volte fast param

In [4]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CON AUTO-RECOVER DA PARQUET)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Ricerca flessibile per evitare KeyError se i nomi dei Tier nel JSON sono estesi
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# --- PROTEZIONE DA NAMEERROR: SE LA RAM È VUOTA, LEGGE DIRETTAMENTE I PARQUET TARGET ---
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 3D Pura Singola) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI (MACD)
# ==============================================================================
fast_range = np.arange(4, 68, 2)     # Fast EMA
slow_range = np.arange(16, 100, 4)    # Slow EMA
sig_range  = np.arange(5, 100, 5)     # Signal Line EMA

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 ASINCRONO + MACD CONVOLUZIONE 3D (SINGLE TOP 1)
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
        
    # CAGR
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
    # Volatilità e Sharpe (Annualizzati)
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
    # Sortino (Annualizzato)
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
    # Max DD
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
    # Scudo contro i prezzi a 0.0 provenienti dai bad-tick di MT5
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    close.index = pd.to_datetime(close.index)
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. GENERAZIONE COMBINAZIONI TRIDIMENSIONALI E VETTORIZZAZIONE IS
    # ==============================================================
    # 🔥 FILTRO ARMONICO: Lo Slow deve essere almeno 1.7 volte il Fast (Disinnesca l'overfitting)
    griglia_3d = [(f, s, sig) for f in fast_range for s in slow_range if s >= 1.7 * f for sig in sig_range]
    f_arr = [c[0] for c in griglia_3d]
    s_arr = [c[1] for c in griglia_3d]
    sig_arr = [c[2] for c in griglia_3d]
    
    macd_is = vbt.MACD.run(close_is, fast_window=f_arr, slow_window=s_arr, signal_window=sig_arr, param_product=False)
    
    # Generazione dei Crossover impulsivi (Stile 3EMA) per proteggere freq='D'
    entries_is = macd_is.macd.vbt.crossed_above(macd_is.signal).vbt.signals.fshift(1)
    exits_is = macd_is.macd.vbt.crossed_below(macd_is.signal).vbt.signals.fshift(1)
    
    pf_is = vbt.Portfolio.from_signals(close_is, entries_is, exits_is, freq='D')
    
    std_is = pf_is.returns().std()
    tpy_is = pf_is.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_is = pf_is.sharpe_ratio()
    
    # Validiamo solo canali stabili con trade sufficienti
    valid_sharpes = np.where((std_is > 0) & (tpy_is >= 2), sharpes_is, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del macd_is, entries_is, exits_is, pf_is
        gc.collect()
        return "LOW_SHARPE"
        
    # ==============================================================
    # 2. COSTRUZIONE CUBO E FASE DI CONVOLUZIONE 3D PURA
    # ==============================================================
    shape_3d = (len(fast_range), len(slow_range), len(sig_range))
    cubo_sharpe = np.full(shape_3d, np.nan)
    
    for idx, (f, s, sig) in enumerate(griglia_3d):
        f_idx = np.where(fast_range == f)[0][0]
        s_idx = np.where(slow_range == s)[0][0]
        sig_idx = np.where(sig_range == sig)[0][0]
        cubo_sharpe[f_idx, s_idx, sig_idx] = valid_sharpes[idx]
        
    convolved_candidates = []
    
    # Scansione tridimensionale dello spazio dei parametri
    for f_idx in range(shape_3d[0]):
        for s_idx in range(shape_3d[1]):
            for sig_idx in range(shape_3d[2]):
                if np.isnan(cubo_sharpe[f_idx, s_idx, sig_idx]):
                    continue
                    
                # Limiti del vicinato cubico 3x3x3 (27 celle complessive)
                f_min, f_max = max(0, f_idx-1), min(shape_3d[0], f_idx+2)
                s_min, s_max = max(0, s_idx-1), min(shape_3d[1], s_idx+2)
                sig_min, sig_max = max(0, sig_idx-1), min(shape_3d[2], sig_idx+2)
                
                vicinato = cubo_sharpe[f_min:f_max, s_min:s_max, sig_min:sig_max].flatten()
                valid_neighbors = vicinato[~np.isnan(vicinato)]
                
                # Richiediamo un altopiano solido: almeno 10 vicini validi su 27 nel cubo
                if len(valid_neighbors) >= 10:
                    conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                    convolved_candidates.append((conv_score, (fast_range[f_idx], slow_range[s_idx], sig_range[sig_idx])))
                    
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    del macd_is, entries_is, exits_is, pf_is
    gc.collect()
    
    if not convolved_candidates:
        return "LOW_SHARPE"
        
    # Estraiamo unicamente il re indiscusso dell'altopiano stabile (Top 1 assoluto)
    best_f, best_s, best_sig = convolved_candidates[0][1]
        
    # ==============================================================
    # 4. CALCOLO STRATEGIA SINGOLA ROBUSTA (IS e OOS)
    # ==============================================================
    macd_final = vbt.MACD.run(close, fast_window=best_f, slow_window=best_s, signal_window=best_sig)
    entries_final = macd_final.macd.vbt.crossed_above(macd_final.signal).vbt.signals.fshift(1)
    exits_final = macd_final.macd.vbt.crossed_below(macd_final.signal).vbt.signals.fshift(1)
    
    # Backtest IS e OOS sulla singola colonna finale
    pf_i = vbt.Portfolio.from_signals(close_is, entries_final.iloc[:split_idx], exits_final.iloc[:split_idx], freq='D')
    is_rets = pf_i.returns()
    trd_is = pf_i.trades.count()
    win_rate_is = ((pf_i.trades.pnl > 0).sum() / trd_is * 100) if trd_is > 0 else 0
    
    pf_o = vbt.Portfolio.from_signals(close_oos, entries_final.iloc[split_idx:], exits_final.iloc[split_idx:], freq='D')
    oos_rets = pf_o.returns()
    trd_oos = pf_o.trades.count()
    win_rate_oos = ((pf_o.trades.pnl > 0).sum() / trd_oos * 100) if trd_oos > 0 else 0
        
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(is_rets, giorni_anno)
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(oos_rets, giorni_anno)
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = f"({int(best_f)},{int(best_s)},{int(best_sig)})"
    
    del macd_final, entries_final, exits_final, pf_i, pf_o
    gc.collect()

    return {
        "Parametri": param_str,
        "Inizio_OOS": data_inizio_oos,
        "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
        "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
        "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
        "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
        "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
        "TRD_IS": trd_is, "TRD_OOS": trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE MACD 3D RIGIDO {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE MACD CONVOLUZIONE 3D (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Robust 3D)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 3D Pura Singola) su 90 asset.

⌛ ELABORAZIONE MACD 3D RIGIDO COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: (14,32,25) | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.95 | Cagr: 88.7% | DD: -33.0% | Sort: 1.90 | Trd: 9.0 | Win: 55.6%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: 0.28 | Cagr: 3.0% | DD: -47.0% | Sort: 0.34 | Trd: 5.0 | Win: 40.0%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  57.01%

⌛ ELABORAZIONE MACD 3D RIGIDO COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: (54,92,50) | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.44 | Cagr: 35.8% | DD: -22.1% | Sort: 1.36 | Trd: 5.0 | Win: 80.0%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2.53
  [OOS] STRAT -> Shp: -0.37 | Cagr: -8.1% | DD: -22.1% | Sort: -0.26 | Trd: 2.0 | Win: 50.0%


# conv. 3d + slow 1.7 fast + segnali generati dallo 0 e non da signal

In [5]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CON AUTO-RECOVER DA PARQUET)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Ricerca flessibile per evitare KeyError se i nomi dei Tier nel JSON sono estesi
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# --- PROTEZIONE DA NAMEERROR: SE LA RAM È VUOTA, LEGGE DIRETTAMENTE I PARQUET TARGET ---
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore MACD Zero-Cross Ibrido (Ingresso Zero / Uscita Signal) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI (MACD)
# ==============================================================================
fast_range = np.arange(4, 68, 2)     # Fast EMA
slow_range = np.arange(16, 100, 4)    # Slow EMA
sig_range  = np.arange(5, 100, 5)     # Signal Line EMA

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 + CONVOLUZIONE 3D + LOGICA IBRIDA ASSE ZERO
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
        
    # CAGR
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
    # Volatilità e Sharpe (Annualizzati)
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
    # Sortino (Annualizzato)
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
    # Max DD
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
    # Scudo contro i prezzi a 0.0 provenienti dai bad-tick di MT5
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    close.index = pd.to_datetime(close.index)
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. GENERAZIONE COMBINAZIONI TRIDIMENSIONALI E VETTORIZZAZIONE IS
    # ==============================================================
    griglia_3d = [(f, s, sig) for f in fast_range for s in slow_range if s >= 1.7 * f for sig in sig_range]
    f_arr = [c[0] for c in griglia_3d]
    s_arr = [c[1] for c in griglia_3d]
    sig_arr = [c[2] for c in griglia_3d]
    
    macd_is = vbt.MACD.run(close_is, fast_window=f_arr, slow_window=s_arr, signal_window=sig_arr, param_product=False)
    
    # 🔥 STRATEGIA IBRIDA: Ingresso quando il MACD taglia lo ZERO, Uscita quando taglia la SIGNAL LINE
    entries_is = macd_is.macd.vbt.crossed_above(0).vbt.signals.fshift(1)
    exits_is = macd_is.macd.vbt.crossed_below(macd_is.signal).vbt.signals.fshift(1)
    
    pf_is = vbt.Portfolio.from_signals(close_is, entries_is, exits_is, freq='D')
    
    std_is = pf_is.returns().std()
    tpy_is = pf_is.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_is = pf_is.sharpe_ratio()
    
    valid_sharpes = np.where((std_is > 0) & (tpy_is >= 2), sharpes_is, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del macd_is, entries_is, exits_is, pf_is
        gc.collect()
        return "LOW_SHARPE"
        
    # ==============================================================
    # 2. COSTRUZIONE CUBO E FASE DI CONVOLUZIONE 3D PURA
    # ==============================================================
    shape_3d = (len(fast_range), len(slow_range), len(sig_range))
    cubo_sharpe = np.full(shape_3d, np.nan)
    
    for idx, (f, s, sig) in enumerate(griglia_3d):
        f_idx = np.where(fast_range == f)[0][0]
        s_idx = np.where(slow_range == s)[0][0]
        sig_idx = np.where(sig_range == sig)[0][0]
        cubo_sharpe[f_idx, s_idx, sig_idx] = valid_sharpes[idx]
        
    convolved_candidates = []
    
    for f_idx in range(shape_3d[0]):
        for s_idx in range(shape_3d[1]):
            for sig_idx in range(shape_3d[2]):
                if np.isnan(cubo_sharpe[f_idx, s_idx, sig_idx]):
                    continue
                    
                f_min, f_max = max(0, f_idx-1), min(shape_3d[0], f_idx+2)
                s_min, s_max = max(0, s_idx-1), min(shape_3d[1], s_idx+2)
                sig_min, sig_max = max(0, sig_idx-1), min(shape_3d[2], sig_idx+2)
                
                vicinato = cubo_sharpe[f_min:f_max, s_min:s_max, sig_min:sig_max].flatten()
                valid_neighbors = vicinato[~np.isnan(vicinato)]
                
                if len(valid_neighbors) >= 10:
                    conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                    convolved_candidates.append((conv_score, (fast_range[f_idx], slow_range[s_idx], sig_range[sig_idx])))
                    
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    del macd_is, entries_is, exits_is, pf_is
    gc.collect()
    
    if not convolved_candidates:
        return "LOW_SHARPE"
        
    # Estraiamo il re indiscusso dell'altopiano stabile (Top 1 assoluto)
    best_f, best_s, best_sig = convolved_candidates[0][1]
        
    # ==============================================================
    # 4. CALCOLO STRATEGIA SINGOLA ROBUSTA (IS e OOS)
    # ==============================================================
    macd_final = vbt.MACD.run(close, fast_window=best_f, slow_window=best_s, signal_window=best_sig)
    
    # 🔥 STRATEGIA IBRIDA ANCHE IN SINOPIA FINALE
    entries_final = macd_final.macd.vbt.crossed_above(0).vbt.signals.fshift(1)
    exits_final = macd_final.macd.vbt.crossed_below(macd_final.signal).vbt.signals.fshift(1)
    
    pf_i = vbt.Portfolio.from_signals(close_is, entries_final.iloc[:split_idx], exits_final.iloc[:split_idx], freq='D')
    is_rets = pf_i.returns()
    trd_is = pf_i.trades.count()
    win_rate_is = ((pf_i.trades.pnl > 0).sum() / trd_is * 100) if trd_is > 0 else 0
    
    pf_o = vbt.Portfolio.from_signals(close_oos, entries_final.iloc[split_idx:], exits_final.iloc[split_idx:], freq='D')
    oos_rets = pf_o.returns()
    trd_oos = pf_o.trades.count()
    win_rate_oos = ((pf_o.trades.pnl > 0).sum() / trd_oos * 100) if trd_oos > 0 else 0
        
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(is_rets, giorni_anno)
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(oos_rets, giorni_anno)
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = f"({int(best_f)},{int(best_s)},{int(best_sig)})"
    
    del macd_final, entries_final, exits_final, pf_i, pf_o
    gc.collect()

    return {
        "Parametri": param_str,
        "Inizio_OOS": data_inizio_oos,
        "SHP_IS": shp_is, "BH_SHP_IS": shp_bh_is, "SHP_OOS": shp_oos, "BH_SHP_OOS": shp_bh_oos,
        "CAGR_IS": cagr_is, "BH_CAGR_IS": cagr_bh_is, "CAGR_OOS": cagr_oos, "BH_CAGR_OOS": cagr_bh_oos,
        "DD_IS": dd_is, "BH_DD_IS": dd_bh_is, "DD_OOS": dd_oos, "BH_DD_OOS": dd_bh_oos,
        "VOL_IS": vol_is, "BH_VOL_IS": vol_bh_is, "VOL_OOS": vol_oos, "BH_VOL_OOS": vol_bh_oos,
        "SORT_IS": sort_is, "BH_SORT_IS": sort_bh_is, "SORT_OOS": sort_oos, "BH_SORT_OOS": sort_bh_oos,
        "TRD_IS": trd_is, "TRD_OOS": trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE MACD ZERO-CROSS IBRIDO {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE MACD ZERO-CROSS IBRIDO (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Robust 3D)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore MACD Zero-Cross Ibrido (Ingresso Zero / Uscita Signal) su 90 asset.

⌛ ELABORAZIONE MACD ZERO-CROSS IBRIDO COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: (4,16,85) | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.62 | Cagr: 71.4% | DD: -49.6% | Sort: 1.64 | Trd: 11.0 | Win: 45.5%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: -0.11 | Cagr: -10.2% | DD: -40.5% | Sort: -0.12 | Trd: 8.0 | Win: 37.5%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  43.78%

⌛ ELABORAZIONE MACD ZERO-CROSS IBRIDO COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: (4,72,5) | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.42 | Cagr: 22.6% | DD: -11.2% | Sort: 0.73 | Trd: 8.0 | Win: 50.0%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2.53
  [OOS] STRAT -> Shp: 0.71 | Cagr: 5.8% | DD: -6.8% | Sort: 0.35 | Trd: 2.0 | 

# conv 3d + distanziamento + ensemble top 2 zero cross

In [5]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CON AUTO-RECOVER DA PARQUET)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Ricerca flessibile per evitare KeyError se i nomi dei Tier nel JSON sono estesi
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# --- PROTEZIONE DA NAMEERROR: SE LA RAM È VUOTA, LEGGE DIRETTAMENTE I PARQUET TARGET ---
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore MACD Zero-Cross Ibrido (Ensemble TOP 2 Distanziato) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI (MACD)
# ==============================================================================
fast_range = np.arange(4, 68, 2)     # Fast EMA
slow_range = np.arange(16, 100, 4)   # Slow EMA
sig_range  = np.arange(5, 100, 5)    # Signal Line EMA

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 + CONVOLUZIONE 3D + ENSEMBLE TOP 2
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    close.index = pd.to_datetime(close.index)
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. GENERAZIONE COMBINAZIONI TRIDIMENSIONALI E VETTORIZZAZIONE IS
    # ==============================================================
    # 🔥 Filtro Armonico
    griglia_3d = [(f, s, sig) for f in fast_range for s in slow_range if s >= 1.7 * f for sig in sig_range]
    f_arr = [c[0] for c in griglia_3d]
    s_arr = [c[1] for c in griglia_3d]
    sig_arr = [c[2] for c in griglia_3d]
    
    macd_is = vbt.MACD.run(close_is, fast_window=f_arr, slow_window=s_arr, signal_window=sig_arr, param_product=False)
    
    # 🔥 Strategia Ibrida: Ingresso Zero-Cross, Uscita Signal Line
    entries_is = macd_is.macd.vbt.crossed_above(0).vbt.signals.fshift(1)
    exits_is = macd_is.macd.vbt.crossed_below(macd_is.signal).vbt.signals.fshift(1)
    
    pf_is = vbt.Portfolio.from_signals(close_is, entries_is, exits_is, freq='D')
    
    std_is = pf_is.returns().std()
    tpy_is = pf_is.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_is = pf_is.sharpe_ratio()
    
    valid_sharpes = np.where((std_is > 0) & (tpy_is >= 2), sharpes_is, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del macd_is, entries_is, exits_is, pf_is
        gc.collect()
        return "LOW_SHARPE"
        
    # ==============================================================
    # 2. COSTRUZIONE CUBO E FASE DI CONVOLUZIONE 3D PURA
    # ==============================================================
    shape_3d = (len(fast_range), len(slow_range), len(sig_range))
    cubo_sharpe = np.full(shape_3d, np.nan)
    
    for idx, (f, s, sig) in enumerate(griglia_3d):
        f_idx = np.where(fast_range == f)[0][0]
        s_idx = np.where(slow_range == s)[0][0]
        sig_idx = np.where(sig_range == sig)[0][0]
        cubo_sharpe[f_idx, s_idx, sig_idx] = valid_sharpes[idx]
        
    convolved_candidates = []
    
    for f_idx in range(shape_3d[0]):
        for s_idx in range(shape_3d[1]):
            for sig_idx in range(shape_3d[2]):
                if np.isnan(cubo_sharpe[f_idx, s_idx, sig_idx]):
                    continue
                    
                f_min, f_max = max(0, f_idx-1), min(shape_3d[0], f_idx+2)
                s_min, s_max = max(0, s_idx-1), min(shape_3d[1], s_idx+2)
                sig_min, sig_max = max(0, sig_idx-1), min(shape_3d[2], sig_idx+2)
                
                vicinato = cubo_sharpe[f_min:f_max, s_min:s_max, sig_min:sig_max].flatten()
                valid_neighbors = vicinato[~np.isnan(vicinato)]
                
                if len(valid_neighbors) >= 10:
                    conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                    convolved_candidates.append((conv_score, (fast_range[f_idx], slow_range[s_idx], sig_range[sig_idx])))
                    
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    del macd_is, entries_is, exits_is, pf_is
    gc.collect()
    
    if not convolved_candidates:
        return "LOW_SHARPE"
        
    # ==============================================================
    # 3. SELEZIONE TOP 2 CON DISTANZA SPAZIALE (>= 15 Periodi)
    # ==============================================================
    DISTANZA_MINIMA = 15
    top_stable_params = []
    
    for score, p_triplet in convolved_candidates:
        f, s, sig = p_triplet
        troppo_vicino = False
        
        for (sf, ss, ssig) in top_stable_params:
            # Distanza Manhattan tra i parametri
            distanza_totale = abs(f - sf) + abs(s - ss) + abs(sig - ssig)
            if distanza_totale < DISTANZA_MINIMA:
                troppo_vicino = True
                break
                
        if not troppo_vicino:
            top_stable_params.append((f, s, sig))
            if len(top_stable_params) == 2:  # Ci fermiamo esattamente a 2
                break
                
    if not top_stable_params:
        return "LOW_SHARPE"

    # ==============================================================
    # 4. CALCOLO ENSEMBLE FINALE (IS e OOS)
    # ==============================================================
    f_top = [p[0] for p in top_stable_params]
    s_top = [p[1] for p in top_stable_params]
    sig_top = [p[2] for p in top_stable_params]
    
    macd_final = vbt.MACD.run(close, fast_window=f_top, slow_window=s_top, signal_window=sig_top, param_product=False)
    
    entries_final = macd_final.macd.vbt.crossed_above(0).vbt.signals.fshift(1)
    exits_final = macd_final.macd.vbt.crossed_below(macd_final.signal).vbt.signals.fshift(1)
    
    is_rets_list, oos_rets_list = [], []
    is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
    for idx in range(len(top_stable_params)):
        pf_i = vbt.Portfolio.from_signals(close_is, entries_final.iloc[:, idx].iloc[:split_idx], exits_final.iloc[:, idx].iloc[:split_idx], freq='D')
        is_rets_list.append(pf_i.returns())
        is_trd.append(pf_i.trades.count())
        is_win.append((pf_i.trades.pnl > 0).sum())
        
        pf_o = vbt.Portfolio.from_signals(close_oos, entries_final.iloc[:, idx].iloc[split_idx:], exits_final.iloc[:, idx].iloc[split_idx:], freq='D')
        oos_rets_list.append(pf_o.returns())
        oos_trd.append(pf_o.trades.count())
        oos_win.append((pf_o.trades.pnl > 0).sum())
        
    # Blended Performance (Media Equiponderata dei 2 Modelli)
    blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
    blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
    avg_trd_is = np.mean(is_trd)
    win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
    avg_trd_oos = np.mean(oos_trd)
    win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
    del macd_final, entries_final, exits_final
    gc.collect()

    # --- 🔥 INSERITI METADATI RAW COMPLETI PER EXPORT SPECCHIATO ---
    return {
        "Parametri": param_str,
        "Parametri_Raw": [[int(p[0]), int(p[1]), int(p[2])] for p in top_stable_params],
        "Giorni_Anno": giorni_anno,
        "Inizio_OOS": data_inizio_oos,
        
        # Strategy Metrics
        "SHP_IS": shp_is, "SHP_OOS": shp_oos,
        "CAGR_IS": cagr_is, "CAGR_OOS": cagr_oos,
        "DD_IS": dd_is, "DD_OOS": dd_oos,
        "VOL_IS": vol_is, "VOL_OOS": vol_oos,
        "SORT_IS": sort_is, "SORT_OOS": sort_oos,
        "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        
        # Benchmark Metrics (Buy & Hold)
        "BH_SHP_IS": shp_bh_is, "BH_SHP_OOS": shp_bh_oos,
        "BH_CAGR_IS": cagr_bh_is, "BH_CAGR_OOS": cagr_bh_oos,
        "BH_DD_IS": dd_bh_is, "BH_DD_OOS": dd_bh_oos,
        "BH_VOL_IS": vol_bh_is, "BH_VOL_OOS": vol_bh_oos,
        "BH_SORT_IS": sort_bh_is, "BH_SORT_OOS": sort_bh_oos,
        
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE MACD ZERO-CROSS IBRIDO (TOP 2) {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trovata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE MACD ZERO-CROSS IBRIDO TOP 2 (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Top 2)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

# ==============================================================================
# 🔥 6. SALVATAGGIO CONFIGURAZIONE JSON PER INTERFACCIA ENSEMBLE
# ==============================================================================
if len(report_finale) > 0:
    export_dir = "progetto_trading_data"
    os.makedirs(export_dir, exist_ok=True)
    path_export = os.path.join(export_dir, "config_macd_zerocross.json")
    
    config_json = {
        "NOME_STRATEGIA": "MACD_ZeroCross_3D_Convolution_Ensemble",
        "DATA_GENERAZIONE": "2026-06-21",
        "ASSETS": {}
    }
    
    for item in report_finale:
        ticker = item["Ticker"]
        config_json["ASSETS"][ticker] = {
            "Cluster": item["Cluster"],
            "Data_Inizio_OOS": item["Inizio_OOS"],
            "Giorni_Anno": int(item["Giorni_Anno"]),
            "Numero_Modelli_Ensemble": len(item["Parametri_Raw"]),
            "Parametri_Ottimi": item["Parametri_Raw"],
            "Alpha_OOS": float(item["Alpha_OOS"]),
            
            "Metrics_In_Sample": {
                "Sharpe_IS": float(item["SHP_IS"]),
                "Sortino_IS": float(item["SORT_IS"]),
                "CAGR_IS": float(item["CAGR_IS"]),
                "Volatilità_IS": float(item["VOL_IS"]),
                "MaxDD_IS": float(item["DD_IS"]),
                "Trades_IS": float(item["TRD_IS"]),
                "WinRate_IS": float(item["WIN_IS"])
            },
            "Metrics_Out_Of_Sample": {
                "Sharpe_OOS": float(item["SHP_OOS"]),
                "Sortino_OOS": float(item["SORT_OOS"]),
                "CAGR_OOS": float(item["CAGR_OOS"]),
                "Volatilità_OOS": float(item["VOL_OOS"]),
                "MaxDD_OOS": float(item["DD_OOS"]),
                "Trades_OOS": float(item["TRD_OOS"]),
                "WinRate_OOS": float(item["WIN_OOS"])
            },
            "Benchmark_In_Sample": {
                "Sharpe_BH_IS": float(item["BH_SHP_IS"]),
                "Sortino_BH_IS": float(item["BH_SORT_IS"]),
                "CAGR_BH_IS": float(item["BH_CAGR_IS"]),
                "Volatilità_BH_IS": float(item["BH_VOL_IS"]),
                "MaxDD_BH_IS": float(item["BH_DD_IS"])
            },
            "Benchmark_Out_Of_Sample": {
                "Sharpe_BH_OOS": float(item["BH_SHP_OOS"]),
                "Sortino_BH_OOS": float(item["BH_SORT_OOS"]),
                "CAGR_BH_OOS": float(item["BH_CAGR_OOS"]),
                "Volatilità_BH_OOS": float(item["BH_VOL_OOS"]),
                "MaxDD_BH_OOS": float(item["BH_DD_OOS"])
            }
        }
        
    with open(path_export, "w") as f:
        json.dump(config_json, f, indent=4)
        
    print(f"\n💾 Configurazione globale salvata con successo in: {path_export} 🚀")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore MACD Zero-Cross Ibrido (Ensemble TOP 2 Distanziato) su 90 asset.

⌛ ELABORAZIONE MACD ZERO-CROSS IBRIDO (TOP 2) COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: [(4,16,85), (10,20,60)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.74 | Cagr: 71.2% | DD: -41.8% | Sort: 1.91 | Trd: 11.0 | Win: 45.5%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: -0.15 | Cagr: -11.3% | DD: -46.7% | Sort: -0.18 | Trd: 6.5 | Win: 38.5%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  42.66%

⌛ ELABORAZIONE MACD ZERO-CROSS IBRIDO (TOP 2) COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: [(4,72,5), (4,84,10)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.49 | Cagr: 21.2% | DD: -9.4% | Sort: 0.79 | Trd: 6.5 | Win: 53.8%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2.53
  [OOS] STRAT -> Shp: 0.86 | Cagr: 7.4%

# conv. 3d + distanziamento + selezione top 2

In [4]:
import os
import json
import numpy as np
import pandas as pd
import vectorbt as vbt
import gc  # Memory Management

# ==============================================================================
# 1. CONFIGURAZIONE E CARICAMENTO MAPPA STRATEGIE (CON AUTO-RECOVER DA PARQUET)
# ==============================================================================
path_mappa = os.path.join("progetto_trading_data", "mappa_strategie.json")
with open(path_mappa, "r") as f:
    mappa_strategie = json.load(f)

# Ricerca flessibile per evitare KeyError se i nomi dei Tier nel JSON sono estesi
tier1_keys = [k for k in mappa_strategie.keys() if "tier 1" in k.lower() or "tier_1" in k.lower()]
tier2_keys = [k for k in mappa_strategie.keys() if "tier 2" in k.lower() or "tier_2" in k.lower()]

ticker_target = []
for k in tier1_keys + tier2_keys:
    ticker_target.extend(mappa_strategie[k])

BASE_DIR = os.path.join("progetto_trading_data", "dataset_v3_parquet")
all_tickers_dict = {}

# --- PROTEZIONE DA NAMEERROR: SE LA RAM È VUOTA, LEGGE DIRETTAMENTE I PARQUET TARGET ---
if 'dati_puliti' in locals() or 'dati_puliti' in globals():
    for cluster, tickers in dati_puliti.items():
        for t, df in tickers.items():
            if t in ticker_target:
                all_tickers_dict[t] = {"df": df, "cluster": cluster}
else:
    print("📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...")
    if os.path.exists(BASE_DIR):
        for cluster in os.listdir(BASE_DIR):
            cluster_path = os.path.join(BASE_DIR, cluster)
            if os.path.isdir(cluster_path):
                for file in os.listdir(cluster_path):
                    if file.endswith(".parquet"):
                        t = file.replace(".parquet", "")
                        if t in ticker_target:
                            all_tickers_dict[t] = {
                                "df": pd.read_parquet(os.path.join(cluster_path, file)),
                                "cluster": cluster
                            }
        if not all_tickers_dict:
            print("⚠️ Attenzione: Nessun file Parquet locale corrisponde ai ticker della mappa_strategie.")
    else:
        print(f"❌ ERRORE CRITICO: La cartella locale {BASE_DIR} non esiste. Impossibile recuperare i dati.")

print(f"🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 3D Pura + Ensemble TOP 2) su {len(all_tickers_dict)} asset.")

# ==============================================================================
# 2. GENERAZIONE RANGE PARAMETRI (MACD)
# ==============================================================================
fast_range = np.arange(4, 68, 2)     # Fast EMA
slow_range = np.arange(16, 100, 4)    # Slow EMA
sig_range  = np.arange(5, 100, 5)     # Signal Line EMA

# ==============================================================================
# 3. CORE FUNCTION: SPLIT 60/40 ASINCRONO + MACD CONVOLUZIONE 3D + ENSEMBLE ENGINE TOP 2
# ==============================================================================

def calc_advanced_metrics(rets, giorni_anno):
    """Funzione helper per calcolare tutte le metriche da una serie di rendimenti"""
    if len(rets) == 0 or rets.isnull().all():
        return 0, 0, 0, 0, 0
        
    # CAGR
    anni = len(rets) / giorni_anno
    cagr = ((1 + rets).prod() ** (1 / anni) - 1) if anni > 0 else 0
    
    # Volatilità e Sharpe (Annualizzati)
    daily_std = rets.std()
    vol = daily_std * np.sqrt(giorni_anno)
    sharpe = (rets.mean() / daily_std * np.sqrt(giorni_anno)) if daily_std > 0 else 0
    
    # Sortino (Annualizzato)
    neg_rets = rets[rets < 0]
    daily_down_std = neg_rets.std()
    sortino = (rets.mean() / daily_down_std * np.sqrt(giorni_anno)) if daily_down_std > 0 else 0
    
    # Max DD
    cum = (1 + rets).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    max_dd = dd.min() if len(dd) > 0 else 0
    
    return cagr, vol, sharpe, sortino, max_dd


def ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=False):
    # Scudo contro i prezzi a 0.0 provenienti dai bad-tick di MT5
    close = df['Close'].replace(0, np.nan).bfill().ffill().copy()
    close.index = pd.to_datetime(close.index)
    
    giorni_anno = 365 if is_crypto else 252
    total_bars = len(close)
    split_idx = int(total_bars * 0.6)
    
    if split_idx < 200 or (total_bars - split_idx) < 50:
        return None
        
    close_is = close.iloc[:split_idx]
    close_oos = close.iloc[split_idx:]
    data_inizio_oos = close_oos.index[0].strftime('%Y-%m-%d')
    anni_is = len(close_is) / giorni_anno
    
    # --- CALCOLO BUY & HOLD IS/OOS ---
    rets_bh_is = close_is.pct_change().dropna()
    cagr_bh_is, vol_bh_is, shp_bh_is, sort_bh_is, dd_bh_is = calc_advanced_metrics(rets_bh_is, giorni_anno)
    
    rets_bh_oos = close_oos.pct_change().dropna()
    cagr_bh_oos, vol_bh_oos, shp_bh_oos, sort_bh_oos, dd_bh_oos = calc_advanced_metrics(rets_bh_oos, giorni_anno)

    # ==============================================================
    # 1. GENERAZIONE COMBINAZIONI TRIDIMENSIONALI E VETTORIZZAZIONE IS
    # ==============================================================
    griglia_3d = [(f, s, sig) for f in fast_range for s in slow_range if f < s for sig in sig_range]
    f_arr = [c[0] for c in griglia_3d]
    s_arr = [c[1] for c in griglia_3d]
    sig_arr = [c[2] for c in griglia_3d]
    
    macd_is = vbt.MACD.run(close_is, fast_window=f_arr, slow_window=s_arr, signal_window=sig_arr, param_product=False)
    
    # Generazione dei Crossover impulsivi (Stile 3EMA) per proteggere freq='D'
    entries_is = macd_is.macd.vbt.crossed_above(macd_is.signal).vbt.signals.fshift(1)
    exits_is = macd_is.macd.vbt.crossed_below(macd_is.signal).vbt.signals.fshift(1)
    
    pf_is = vbt.Portfolio.from_signals(close_is, entries_is, exits_is, freq='D')
    
    std_is = pf_is.returns().std()
    tpy_is = pf_is.trades.count() / anni_is if anni_is > 0 else 0
    sharpes_is = pf_is.sharpe_ratio()
    
    # Validiamo solo canali stabili con trade sufficienti
    valid_sharpes = np.where((std_is > 0) & (tpy_is >= 2), sharpes_is, np.nan)
    
    if np.isnan(valid_sharpes).all():
        del macd_is, entries_is, exits_is, pf_is
        gc.collect()
        return "LOW_SHARPE"
        
    # ==============================================================
    # 2. COSTRUZIONE CUBO E FASE DI CONVOLUZIONE 3D PURA
    # ==============================================================
    shape_3d = (len(fast_range), len(slow_range), len(sig_range))
    cubo_sharpe = np.full(shape_3d, np.nan)
    
    for idx, (f, s, sig) in enumerate(griglia_3d):
        f_idx = np.where(fast_range == f)[0][0]
        s_idx = np.where(slow_range == s)[0][0]
        sig_idx = np.where(sig_range == sig)[0][0]
        cubo_sharpe[f_idx, s_idx, sig_idx] = valid_sharpes[idx]
        
    convolved_candidates = []
    
    # Scansione tridimensionale dello spazio dei parametri
    for f_idx in range(shape_3d[0]):
        for s_idx in range(shape_3d[1]):
            for sig_idx in range(shape_3d[2]):
                if np.isnan(cubo_sharpe[f_idx, s_idx, sig_idx]):
                    continue
                    
                # Limiti del vicinato cubico 3x3x3 (27 celle complessive)
                f_min, f_max = max(0, f_idx-1), min(shape_3d[0], f_idx+2)
                s_min, s_max = max(0, s_idx-1), min(shape_3d[1], s_idx+2)
                sig_min, sig_max = max(0, sig_idx-1), min(shape_3d[2], sig_idx+2)
                
                vicinato = cubo_sharpe[f_min:f_max, s_min:s_max, sig_min:sig_max].flatten()
                valid_neighbors = vicinato[~np.isnan(vicinato)]
                
                # Richiediamo un altopiano solido: almeno 10 vicini validi su 27 nel cubo
                if len(valid_neighbors) >= 10:
                    conv_score = np.mean(valid_neighbors) - np.std(valid_neighbors)
                    convolved_candidates.append((conv_score, (fast_range[f_idx], slow_range[s_idx], sig_range[sig_idx])))
                    
    convolved_candidates.sort(key=lambda x: x[0], reverse=True)
    
    # ==============================================================
    # 3. DISTANZIAMENTO SPAZIALE 3D (MANHATTAN DISTANCE >= 15) -> MODIFICATO PER TOP 2
    # ==============================================================
    DISTANZA_MINIMA = 15
    top_stable_params = []
    
    for score, p_triplet in convolved_candidates:
        f, s, sig = p_triplet
        comm_too_close = False
        
        for (sf, ss, ssig) in top_stable_params:
            # Distanza geometrica combinata tra i tre assi parametrici
            distanza_totale = abs(f - sf) + abs(s - ss) + abs(sig - ssig)
            if  distanza_totale < DISTANZA_MINIMA:
                comm_too_close = True
                break
                
        if not comm_too_close:
            top_stable_params.append((f, s, sig))
            # Ci fermiamo esattamente ai 2 migliori picchi indipendenti
            if len(top_stable_params) == 2:  
                break
                
    del macd_is, entries_is, exits_is, pf_is
    gc.collect()
    
    if not top_stable_params:
        return "LOW_SHARPE"
        
    # ==============================================================
    # 4. CALCOLO ENSEMBLE FINALE CROSS-ASSET (1-TO-1 MAPPING)
    # ==============================================================
    f_top = [p[0] for p in top_stable_params]
    s_top = [p[1] for p in top_stable_params]
    sig_top = [p[2] for p in top_stable_params]
    
    # param_product=False accoppia gli indici generando esattamente 2 colonne (Top 2)
    macd_final = vbt.MACD.run(close, fast_window=f_top, slow_window=s_top, signal_window=sig_top, param_product=False)
    entries_final = macd_final.macd.vbt.crossed_above(macd_final.signal).vbt.signals.fshift(1)
    exits_final = macd_final.macd.vbt.crossed_below(macd_final.signal).vbt.signals.fshift(1)
    
    is_rets_list, oos_rets_list = [], []
    is_trd, is_win, oos_trd, oos_win = [], [], [], []
    
    for idx in range(len(top_stable_params)):
        # Elaborazione posizionale per evitare i conflitti di MultiIndex delle 2 colonne
        pf_i = vbt.Portfolio.from_signals(close_is, entries_final.iloc[:, idx].iloc[:split_idx], exits_final.iloc[:, idx].iloc[:split_idx], freq='D')
        is_rets_list.append(pf_i.returns())
        is_trd.append(pf_i.trades.count())
        is_win.append((pf_i.trades.pnl > 0).sum())
        
        pf_o = vbt.Portfolio.from_signals(close_oos, entries_final.iloc[:, idx].iloc[split_idx:], exits_final.iloc[:, idx].iloc[split_idx:], freq='D')
        oos_rets_list.append(pf_o.returns())
        oos_trd.append(pf_o.trades.count())
        oos_win.append((pf_o.trades.pnl > 0).sum())
        
    blended_is_rets = pd.concat(is_rets_list, axis=1).mean(axis=1)
    blended_oos_rets = pd.concat(oos_rets_list, axis=1).mean(axis=1)
    
    cagr_is, vol_is, shp_is, sort_is, dd_is = calc_advanced_metrics(blended_is_rets, giorni_anno)
    avg_trd_is = np.mean(is_trd)
    win_rate_is = (np.sum(is_win) / np.sum(is_trd) * 100) if np.sum(is_trd) > 0 else 0
    
    cagr_oos, vol_oos, shp_oos, sort_oos, dd_oos = calc_advanced_metrics(blended_oos_rets, giorni_anno)
    avg_trd_oos = np.mean(oos_trd)
    win_rate_oos = (np.sum(oos_win) / np.sum(oos_trd) * 100) if np.sum(oos_trd) > 0 else 0
    
    alpha_oos = cagr_oos - cagr_bh_oos
    param_str = "[" + ", ".join([f"({int(p[0])},{int(p[1])},{int(p[2])})" for p in top_stable_params]) + "]"
    
    del macd_final, entries_final, exits_final
    gc.collect()

    # --- 🔥 INSERITI METADATI RAW COMPLETI PER EXPORT SPECCHIATO ---
    return {
        "Parametri": param_str,
        "Parametri_Raw": [[int(p[0]), int(p[1]), int(p[2])] for p in top_stable_params],
        "Giorni_Anno": giorni_anno,
        "Inizio_OOS": data_inizio_oos,
        
        # Strategy Metrics
        "SHP_IS": shp_is, "SHP_OOS": shp_oos,
        "CAGR_IS": cagr_is, "CAGR_OOS": cagr_oos,
        "DD_IS": dd_is, "DD_OOS": dd_oos,
        "VOL_IS": vol_is, "VOL_OOS": vol_oos,
        "SORT_IS": sort_is, "SORT_OOS": sort_oos,
        "TRD_IS": avg_trd_is, "TRD_OOS": avg_trd_oos,
        "WIN_IS": win_rate_is, "WIN_OOS": win_rate_oos,
        
        # Benchmark Metrics (Buy & Hold)
        "BH_SHP_IS": shp_bh_is, "BH_SHP_OOS": shp_bh_oos,
        "BH_CAGR_IS": cagr_bh_is, "BH_CAGR_OOS": cagr_bh_oos,
        "BH_DD_IS": dd_bh_is, "BH_DD_OOS": dd_bh_oos,
        "BH_VOL_IS": vol_bh_is, "BH_VOL_OOS": vol_bh_oos,
        "BH_SORT_IS": sort_bh_is, "BH_SORT_OOS": sort_bh_oos,
        
        "Alpha_OOS": alpha_oos
    }

# ==============================================================================
# 4. LOOP DI ELABORAZIONE GLOBALE CROSS-ASSET
# ==============================================================================
report_finale = []

for ticker, info in all_tickers_dict.items():
    df = info["df"]
    cluster_appartenenza = info["cluster"]
    is_crypto = (cluster_appartenenza.lower() == "crypto_cfd")
    
    print(f"\n============================================================")
    print(f"⌛ ELABORAZIONE MACD 3D TOP 2 {ticker:<8} | Cluster: {cluster_appartenenza}")
    
    metrics = ottimizza_split_60_40_macd_ancorato(ticker, df, fast_range, slow_range, sig_range, is_crypto=is_crypto)
    
    if isinstance(metrics, dict):
        print(f"✅ Parametri Scelti: {metrics['Parametri']} | Inizio OOS: {metrics['Inizio_OOS']}")
        print(f"  [IS]  STRAT -> Shp: {metrics['SHP_IS']:.2f} | Cagr: {metrics['CAGR_IS']*100:.1f}% | DD: {metrics['DD_IS']*100:.1f}% | Sort: {metrics['SORT_IS']:.2f} | Trd: {metrics['TRD_IS']:.1f} | Win: {metrics['WIN_IS']:.1f}%")
        print(f"  [IS]  B&H   -> Shp: {metrics['BH_SHP_IS']:.2f} | Cagr: {metrics['BH_CAGR_IS']*100:.1f}% | DD: {metrics['BH_DD_IS']*100:.1f}% | Sort: {metrics['BH_SORT_IS']:.2f}")
        print(f"  [OOS] STRAT -> Shp: {metrics['SHP_OOS']:.2f} | Cagr: {metrics['CAGR_OOS']*100:.1f}% | DD: {metrics['DD_OOS']*100:.1f}% | Sort: {metrics['SORT_OOS']:.2f} | Trd: {metrics['TRD_OOS']:.1f} | Win: {metrics['WIN_OOS']:.1f}%")
        print(f"  [OOS] B&H   -> Shp: {metrics['BH_SHP_OOS']:.2f} | Cagr: {metrics['BH_CAGR_OOS']*100:.1f}% | DD: {metrics['BH_DD_OOS']*100:.1f}% | Sort: {metrics['BH_SORT_OOS']:.2f}")
        print(f"  🔥 ALPHA OOS:  {metrics['Alpha_OOS']*100:.2f}%")
        
        metrics["Ticker"] = ticker
        metrics["Cluster"] = cluster_appartenenza
        report_finale.append(metrics)
    elif metrics == "LOW_SHARPE":
        print("⚠️ SCARTATO: Nessuna zona valida trouvata In-Sample.")
    else:
        print("❌ ERRORE: Storico dati insufficiente.")

# ==============================================================================
# 5. LEADERBOARD FINALE
# ==============================================================================
if len(report_finale) > 0:
    df_leaderboard = pd.DataFrame(report_finale)
    df_leaderboard = df_leaderboard.sort_values(by="Alpha_OOS", ascending=False).reset_index(drop=True)

    print("\n" + "="*160)
    print("🏆 TABELLONE FINALE MACD ENSEMBLE TOP 2 (Ordinato per Alpha OOS)")
    print("="*160)

    tabella_stampa = df_leaderboard[[
        "Ticker", "Cluster", "Inizio_OOS", "Parametri", 
        "SHP_IS", "BH_SHP_IS", "SHP_OOS", "BH_SHP_OOS", 
        "CAGR_OOS", "BH_CAGR_OOS", "Alpha_OOS", "DD_OOS", "BH_DD_OOS", "TRD_OOS"
    ]].copy()

    tabella_stampa.columns = [
        "TICKER", "CLUSTER", "INIZIO OOS", "PARAMS (Ensemble Top 2)", 
        "SHP IS", "B&H IS", "SHP OOS", "B&H OOS", 
        "CAGR OOS", "B&H CAGR", "ALPHA %", "DD OOS", "B&H DD", "TRD OOS"
    ]

    print(tabella_stampa.to_string(
        index=False, 
        justify='left',
        col_space=9,
        formatters={
            'SHP IS':    '{:,.2f}'.format,
            'B&H IS':    '{:,.2f}'.format,
            'SHP OOS':   '{:,.2f}'.format,
            'B&H OOS':   '{:,.2f}'.format,
            'CAGR OOS':  lambda x: f"{x*100:,.1f}%",
            'B&H CAGR':  lambda x: f"{x*100:,.1f}%",
            'ALPHA %':   lambda x: f"{x*100:,.1f}%",
            'DD OOS':    lambda x: f"{x*100:,.1f}%",
            'B&H DD':    lambda x: f"{x*100:,.1f}%",
            'TRD OOS':   '{:,.1f}'.format
        }
    ))
    print("="*160)
else:
    print("\n⚠️ Nessun asset ha superato i filtri operativi.")

# ==============================================================================
# 🔥 6. SALVATAGGIO CONFIGURAZIONE JSON PER INTERFACCIA ENSEMBLE
# ==============================================================================
if len(report_finale) > 0:
    export_dir = "progetto_trading_data"
    os.makedirs(export_dir, exist_ok=True)
    path_export = os.path.join(export_dir, "config_macd_normale.json")
    
    config_json = {
        "NOME_STRATEGIA": "MACD_Normale_3D_Convolution_Ensemble",
        "DATA_GENERAZIONE": "2026-06-21",
        "ASSETS": {}
    }
    
    for item in report_finale:
        ticker = item["Ticker"]
        config_json["ASSETS"][ticker] = {
            "Cluster": item["Cluster"],
            "Data_Inizio_OOS": item["Inizio_OOS"],
            "Giorni_Anno": int(item["Giorni_Anno"]),
            "Numero_Modelli_Ensemble": len(item["Parametri_Raw"]),
            "Parametri_Ottimi": item["Parametri_Raw"],
            "Alpha_OOS": float(item["Alpha_OOS"]),
            
            "Metrics_In_Sample": {
                "Sharpe_IS": float(item["SHP_IS"]),
                "Sortino_IS": float(item["SORT_IS"]),
                "CAGR_IS": float(item["CAGR_IS"]),
                "Volatilità_IS": float(item["VOL_IS"]),
                "MaxDD_IS": float(item["DD_IS"]),
                "Trades_IS": float(item["TRD_IS"]),
                "WinRate_IS": float(item["WIN_IS"])
            },
            "Metrics_Out_Of_Sample": {
                "Sharpe_OOS": float(item["SHP_OOS"]),
                "Sortino_OOS": float(item["SORT_OOS"]),
                "CAGR_OOS": float(item["CAGR_OOS"]),
                "Volatilità_OOS": float(item["VOL_OOS"]),
                "MaxDD_OOS": float(item["DD_OOS"]),
                "Trades_OOS": float(item["TRD_OOS"]),
                "WinRate_OOS": float(item["WIN_OOS"])
            },
            "Benchmark_In_Sample": {
                "Sharpe_BH_IS": float(item["BH_SHP_IS"]),
                "Sortino_BH_IS": float(item["BH_SORT_IS"]),
                "CAGR_BH_IS": float(item["BH_CAGR_IS"]),
                "Volatilità_BH_IS": float(item["BH_VOL_IS"]),
                "MaxDD_BH_IS": float(item["BH_DD_IS"])
            },
            "Benchmark_Out_Of_Sample": {
                "Sharpe_BH_OOS": float(item["BH_SHP_OOS"]),
                "Sortino_BH_OOS": float(item["BH_SORT_OOS"]),
                "CAGR_BH_OOS": float(item["BH_CAGR_OOS"]),
                "Volatilità_BH_OOS": float(item["BH_VOL_OOS"]),
                "MaxDD_BH_OOS": float(item["BH_DD_OOS"])
            }
        }
        
    with open(path_export, "w") as f:
        json.dump(config_json, f, indent=4)
        
    print(f"\n💾 Configurazione globale salvata con successo in: {path_export} 🚀")

📦 'dati_puliti' non rilevata in RAM. Estrazione dinamica dei ticker target dai file Parquet...
🚀 Motore MACD Split 60/40 Asincrono (Convoluzione 3D Pura + Ensemble TOP 2) su 90 asset.

⌛ ELABORAZIONE MACD 3D TOP 2 COCOA.c  | Cluster: cash_cfd
✅ Parametri Scelti: [(14,32,25), (16,40,20)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 2.00 | Cagr: 90.9% | DD: -29.6% | Sort: 1.99 | Trd: 8.0 | Win: 62.5%
  [IS]  B&H   -> Shp: 1.67 | Cagr: 102.3% | DD: -45.7% | Sort: 2.00
  [OOS] STRAT -> Shp: 0.34 | Cagr: 5.6% | DD: -43.7% | Sort: 0.42 | Trd: 5.0 | Win: 40.0%
  [OOS] B&H   -> Shp: -1.21 | Cagr: -54.0% | DD: -73.9% | Sort: -1.86
  🔥 ALPHA OOS:  59.56%

⌛ ELABORAZIONE MACD 3D TOP 2 COFFEE.c | Cluster: cash_cfd
✅ Parametri Scelti: [(56,76,95), (62,72,85)] | Inizio OOS: 2025-02-03
  [IS]  STRAT -> Shp: 1.96 | Cagr: 50.5% | DD: -12.1% | Sort: 2.08 | Trd: 5.5 | Win: 90.9%
  [IS]  B&H   -> Shp: 1.50 | Cagr: 53.4% | DD: -25.4% | Sort: 2.53
  [OOS] STRAT -> Shp: -0.08 | Cagr: -3.3% | DD: -18.3% | S